# 08 — Gộp L0+L1+L2 vào MỘT file `sdc_iden.onnx`

Notebook này lấy bundle closed-set đã train (`Models/*_field_closedset/model.joblib`) và
nung **toàn bộ** đường quyết định vào một file ONNX duy nhất, để lần deploy xuống router
chỉ còn copy một file.

Graph trả đúng một tensor `string [N,3]`, cột theo thứ tự `make`, `type`, `model`:

```
output[0] = ["Generic Laptop", "Laptop", "Windows Laptop HP"]
```

Không có confidence, không topk, không retrieval. Xác suất và tầng nào trả lời chỉ tồn
tại bên trong graph. Hệ quả: bảng ngưỡng abstain được biểu đạt bằng chính nhãn — dưới
ngưỡng thì head trả `__unknown__`, nên núm tune vẫn sống, đổi lại không phân biệt được
"rừng tự nói không biết" với "có đoán nhưng không đủ tin".

| | `sdc-closedset-onnx-v1` (bản cũ) | `sdc-iden-onnx-v1` (notebook này) |
|---|---|---|
| input | `string [N,40]` | `string [N,44]` + `thresholds float[3,5]` (tuỳ chọn) |
| output | `string [N,3]` — `make`/`type`/`model` | `string [N,3]` — **giữ nguyên hình dạng** |
| L0 rule | không có | OUI, mDNS `model=`, regex hostname — initializer |
| L1 vân tay | không có | 3 mode × 3 head, `LabelEncoder` — initializer |
| hierarchy | không có | ma trận mask — initializer |
| ngưỡng abstain | không có | initializer, **ghi đè được lúc chạy** |
| file xuống router | 1 | 1 |

Hai thứ **không** vào được graph và ở lại dạng code, mãi mãi: `extract_records()`/
`aggregate()` (đọc pcap) và `DeviceTracker` (gộp cửa sổ theo MAC — graph ONNX là
stateless).

Notebook chỉ **đọc** `Data/sessions_verified.parquet` và bundle đã train; nó không train
lại gì và không sửa file nào của pipeline 01–07.

In [1]:
import hashlib
import json
import re
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from onnx import TensorProto as TP
from onnx import compose, helper
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

warnings.filterwarnings("ignore", message=".*InconsistentVersion.*")

_cwd = Path.cwd().resolve()
ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "Data" / "sessions_verified.parquet").is_file()), None)
assert ROOT is not None, f"Không thấy Data/sessions_verified.parquet quanh {_cwd}"
DATA = ROOT / "Data"
MODELS = ROOT / "Models"
SESSIONS_PATH = DATA / "sessions_verified.parquet"
print("ROOT =", ROOT)

ROOT = D:\01.AI_Security\02.SDC


## Hằng số contract

Ba op quyết định việc gộp được vào một file, và cả ba đều có sẵn trong opset 20 mà
project đang dùng:

| Op | Domain | Dùng cho |
|---|---|---|
| `LabelEncoder` | `ai.onnx.ml` | mọi bảng tra: vân tay L1, OUI, mDNS `model=` |
| `StringConcat` | `ai.onnx` (opset 20) | nối khoá vân tay `dhcp_prl \|\| dhcp_vci \|\| tls_fp` |
| `RegexFullMatch` | `ai.onnx` (opset 20) | luật hostname `DESKTOP-[A-Z0-9]{7}` |

`RegexFullMatch` là op đáng chú ý nhất — nó khớp **toàn** chuỗi, cú pháp RE2 (không
lookahead, không backreference). Có nó thì bảng luật hostname không cần nằm ngoài file.

In [2]:
FORMAT = "sdc-iden-onnx-v1"
CONTRACT_VERSION = "1.0.0"
ONNX_FILE = "sdc_iden.onnx"

ONNX_OPSET = 20          # StringConcat / RegexFullMatch xuất hiện ở opset 20
ONNX_ML_OPSET = 3        # ai.onnx.ml LabelEncoder
IR_VERSION = 10
ONNX_LOCALE = "C"        # thiếu locale -> onnxruntime dựng en_US.UTF-8 và chết trên musl

MISSING = "<missing>"
UNKNOWN_LABEL = "__unknown__"
KEY_SEP = " || "
NOT_PRESENT = ("", MISSING, "nan", "None")

HEADS = ("make", "type", "model")
MODEL_HEAD = "model"

IN_FEATURES = "input"
IN_THRESHOLDS = "thresholds"

# Output: đúng MỘT tensor `string [N,3]`, cột theo thứ tự HEADS.
#     output[:, 0] = make   "Generic Laptop"
#     output[:, 1] = type   "Laptop"
#     output[:, 2] = model  "Windows Laptop HP"
# Không trả confidence / topk / retrieval. Xác suất, thứ hạng và tầng nào trả lời đều
# chỉ tồn tại BÊN TRONG graph.
OUT_LABELS = "output"

# Hệ quả của việc bỏ output `abstain`: bảng ngưỡng sẽ thành đồ chết nếu không có chỗ
# nào biểu đạt "không đủ tin". Nên abstain được ghi thẳng vào nhãn: dưới ngưỡng thì head
# trả `__unknown__`. Núm tune ngưỡng vẫn sống, đổi lại mất khả năng phân biệt "rừng tự
# nói không biết" với "có đoán nhưng không đủ tin" — hai thứ mà contract closedset cũ cố
# ý tách. Muốn tách lại thì phải thêm output thứ tư.
ABSTAIN_LABEL = UNKNOWN_LABEL

# Phá hoà xếp hạng. Xác suất của rừng có RẤT nhiều giá trị bằng nhau (lớp không lá nào
# bầu đều được 0.0), và ArgMax của onnxruntime với `argmax` của numpy không bảo đảm phá
# hoà giống nhau. Trừ đi một lượng tăng dần theo chỉ số lớp trước khi xếp hạng biến "ai
# thắng khi hoà" thành luật của MÌNH (chỉ số nhỏ thắng), không còn phụ thuộc bản
# onnxruntime trên router. Nhãn trả ra không đổi vì khoản trừ chỉ dùng để xếp hạng.
TIEBREAK = 1e-6

LAYER_L0, LAYER_L1, LAYER_L2 = 0, 1, 2
LAYER_NAMES = {LAYER_L0: "L0_rule", LAYER_L1: "L1_fingerprint", LAYER_L2: "L2_forest"}

L0_COLUMNS = ("mac_oui", "mac_is_random", "dhcp_hostname", "mdns_model")
SOURCE_FLAGS = ("has_dhcp", "has_dns", "has_mdns", "has_tls")
N_SOURCE_BUCKETS = len(SOURCE_FLAGS) + 1

DEFAULT_THRESHOLDS = {
    "make":  [1.01, 0.75, 0.65, 0.60, 0.55],
    "type":  [1.01, 0.70, 0.60, 0.52, 0.48],
    "model": [1.01, 0.80, 0.70, 0.62, 0.56],
}


def sha256_list(values):
    return hashlib.sha256("\n".join(str(v) for v in values).encode("utf-8")).hexdigest()

## 4 cột L0 nối thêm (40 → 44)

Encoder **không** thấy bốn cột này; chỉ tầng L0 đọc chúng.

| Cột | Kiểu | Nguồn |
|---|---|---|
| `mac_oui` | chuỗi | 3 octet đầu của MAC, `<missing>` nếu MAC ngẫu nhiên hoá |
| `mac_is_random` | số | bit U/L (`0x02`) của octet đầu |
| `dhcp_hostname` | chuỗi | DHCP option 12 |
| `mdns_model` | chuỗi | `model=` trong mDNS TXT `_device-info._tcp` |

Hai cột cuối **extractor hiện tại chưa sinh**. Graph coi `<missing>` là khoá tra trượt
nên model chạy đúng ngay hôm nay, rồi tốt lên khi extractor được bổ sung — không cần
đổi contract lần nữa.

`fe:db:aa:88:5b:35` có `0xfe & 0x02 = 2` → MAC ngẫu nhiên hoá → OUI vô nghĩa. Đây là lý
do iPhone/Android đời mới không tra được OUI: private MAC bật mặc định từ 2020.

In [3]:
_MAC_RE = re.compile(r"^[0-9a-f]{2}(:[0-9a-f]{2}){5}$")


def normalize_mac(value):
    if value is None:
        return MISSING
    text = str(value).strip().lower().replace("-", ":")
    return text if _MAC_RE.match(text) else MISSING


def mac_is_random(value):
    """Bit U/L (0x02) của octet đầu: 1 = locally administered = MAC ngẫu nhiên hoá."""
    mac = normalize_mac(value)
    return 0 if mac == MISSING else int(int(mac[:2], 16) & 0x02 != 0)


def mac_oui(value):
    """3 octet đầu. MAC ngẫu nhiên hoá trả <missing> — OUI của nó vô nghĩa."""
    mac = normalize_mac(value)
    if mac == MISSING or mac_is_random(mac):
        return MISSING
    return mac[:8]


def _clean_text(value):
    if value is None:
        return MISSING
    text = str(value).strip().strip(".")
    return text if text and text not in NOT_PRESENT else MISSING


def add_l0_columns(frame):
    """Bổ sung 4 cột L0. Cột nào extractor chưa sinh thì <missing> — graph tra trượt."""
    out = frame.copy()
    macs = out["mac"] if "mac" in out.columns else pd.Series(MISSING, index=out.index)
    out["mac_oui"] = [mac_oui(v) for v in macs]
    out["mac_is_random"] = [mac_is_random(v) for v in macs]
    for col in ("dhcp_hostname", "mdns_model"):
        out[col] = [_clean_text(v) for v in out[col]] if col in out.columns else MISSING
    return out


def to_input_matrix(frame, columns):
    """frame -> object array [N, len(columns)] toàn chuỗi, đúng thứ tự contract."""
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(f"thiếu cột: {missing}")
    out = frame[list(columns)].copy()
    for col in columns:
        v = out[col]
        out[col] = v.astype(np.float32).astype(str) if pd.api.types.is_numeric_dtype(v) else v.astype(str)
    return out.to_numpy(dtype=object)

## L0 — bằng chứng tất định, confidence 1.0

Ưu tiên **mDNS `model=` > OUI > regex hostname**: `model=iPhone17,1` nói thẳng đời máy
(thứ mà DHCP PRL và TLS fingerprint không nói được vì iPhone 15/16 dùng chung stack);
OUI chỉ nói được hãng và im lặng khi MAC ngẫu nhiên hoá; hostname đứng cuối vì người
dùng đổi được.

Bảng OUI **đào từ chính dataset**, không dùng DB IEEE đầy đủ: IEEE map OUI → tên công ty
sản xuất NIC, không phải nhãn `make` trong taxonomy của project (`Linova/Linux`,
`Generic Laptop` không tồn tại trong IEEE), và 500 KB initializer cho bảng mà 99% dòng
không bao giờ khớp là lãng phí. OUI ứng nhiều nhãn bị **loại hẳn** — trả lời sai ở tầng
conf 1.0 là lỗi im lặng không tầng nào chặn được.

In [4]:
DEFAULT_HOSTNAME_RULES = [
    (r"(?i)desktop-.+", "type", "Laptop"),
    (r"(?i)laptop-.+", "type", "Laptop"),
    (r"(?i).*iphone", "make", "Apple"),
    (r"(?i).*iphone", "type", "Smartphone"),
    (r"(?i).*iphone", "model", "iPhone"),
    (r"(?i).*ipad", "make", "Apple"),
    (r"(?i).*macbook.*", "make", "Apple"),
    (r"(?i)raspberrypi[0-9-]*", "make", "Raspberry Pi"),
    (r"(?i)raspberrypi[0-9-]*", "type", "Single-board Computer"),
    (r"(?i)raspberrypi[0-9-]*", "model", "Raspberry Pi"),
    (r"(?i)android-[0-9a-f]{16}", "type", "Smartphone"),
    (r"(?i)(redmi|xiaomi|poco)[-_ ].*", "make", "Xiaomi"),
    (r"(?i)(galaxy|samsung)[-_ ].*", "make", "Samsung"),
    (r"(?i)oppo[-_ ].*", "make", "OPPO"),
]

DEFAULT_MDNS_MODEL_MAP = {
    "iPhone17,1": "iPhone", "iPhone17,2": "iPhone", "iPhone17,3": "iPhone",
    "iPhone17,4": "iPhone", "iPhone16,1": "iPhone", "iPhone16,2": "iPhone",
    "iPhone15,2": "iPhone", "iPhone15,3": "iPhone", "iPhone14,7": "iPhone",
}


def mine_oui_table(frame, head="make", min_macs=1):
    """OUI -> nhãn, chỉ giữ OUI ứng đúng MỘT nhãn. Nhập nhằng thì loại hẳn."""
    usable = frame[["mac", head]].copy()
    usable["oui"] = [mac_oui(m) for m in usable["mac"]]
    usable = usable[usable["oui"].ne(MISSING)]
    if usable.empty:
        return {}
    stats = usable.groupby("oui").agg(n_label=(head, "nunique"), label=(head, "first"),
                                      n_mac=("mac", "nunique"))
    pure = stats[(stats.n_label == 1) & (stats.n_mac >= min_macs)]
    return {str(k): str(v) for k, v in pure.label.items()}


def build_rules(frame, extra_mdns=None):
    rules = {"oui": mine_oui_table(frame),
             "mdns_model": dict(DEFAULT_MDNS_MODEL_MAP),
             "hostname": [list(r) for r in DEFAULT_HOSTNAME_RULES]}
    if extra_mdns:
        rules["mdns_model"].update(extra_mdns)
    for pattern, _, _ in rules["hostname"]:
        re.compile(pattern)
    return rules


def resolve_rules(rules, labels):
    """Nối nhãn -> chỉ số lớp của từng head. Nhãn không có trong head thì BỎ, có báo."""
    index = {h: {str(c): i for i, c in enumerate(cs)} for h, cs in labels.items()}
    dropped = []

    def idx(head, label):
        got = index.get(head, {}).get(str(label))
        if got is None:
            dropped.append(f"{head}={label!r}")
        return got

    oui = {}
    for key, label in rules["oui"].items():
        got = idx("make", label)
        if got is not None:
            oui[key] = got
    mdns = {}
    for key, label in rules["mdns_model"].items():
        got = idx(MODEL_HEAD, label)
        if got is not None:
            mdns[key] = got
    hostname = []
    for pattern, head, label in rules["hostname"]:
        got = idx(head, label)
        if got is not None:
            hostname.append((pattern, head, got))
    return {"oui": oui, "mdns_model": mdns, "hostname": hostname,
            "dropped": sorted(set(dropped))}

## L1 — bảng vân tay exact

Phản chiếu `FingerprintTable` ở `03_train_model.ipynb`: cùng 3 mode, cùng gác
`min_support=3` / `min_devices=2`, cùng thứ tự tra cụ thể → chung.

`LabelEncoder` chỉ trả được một số nguyên, nên ba trạng thái của `lookup()` mã hoá vào
chính giá trị đó:

| Giá trị | Nghĩa | Hành vi |
|---|---|---|
| `>= 0` | hit | trả lời, dừng chuỗi |
| `-1` | miss | rơi xuống mode chung hơn |
| `-2` | ambiguous | **không** trả lời và **không** rơi tầng |

Trạng thái `-2` phải có thật. Vứt nó đi rồi để `fp_full` nhập nhằng rơi xuống `fp_dhcp`
là biến câu "vân tay này không tách được nhãn" thành một câu trả lời sai với confidence
1.0 — đúng cái lỗi mà `min_devices=2` sinh ra để chặn.

In [5]:
HIT_MISS, HIT_AMBIGUOUS = -1, -2

FP_MODES = (
    ("fp_full", ("dhcp_prl", "dhcp_vci", "tls_fp"), ("has_dhcp", "has_tls")),
    ("fp_dhcp", ("dhcp_prl", "dhcp_vci"), ("has_dhcp",)),
    ("fp_tls", ("tls_fp",), ("has_tls",)),
)
DEVICE_COL = "canonical_device"


def _present_series(values):
    text = values.astype(str)
    mask = text.str.strip().ne("")
    for bad in NOT_PRESENT:
        mask &= text.ne(bad)
    return mask


def fp_usable(frame, cols, flags):
    mask = pd.Series(True, index=frame.index)
    for flag in flags:
        mask &= frame[flag].astype(float).eq(1.0)
    for col in cols:
        mask &= _present_series(frame[col])
    return mask


def fp_key(frame, cols):
    return frame[list(cols)].astype(str).agg(KEY_SEP.join, axis=1)


def mine_fingerprints(frame, labels, min_support=3, min_devices=2):
    """{mode: {head: {key: value}}}; value >=0 là lớp, -2 là nhập nhằng."""
    index = {h: {str(c): i for i, c in enumerate(cs)} for h, cs in labels.items()}
    tables = {}
    for mode, cols, flags in FP_MODES:
        sub = frame[fp_usable(frame, cols, flags)]
        for head in labels:
            book = {}
            if not sub.empty:
                stats = (pd.DataFrame({"k": fp_key(sub, cols).to_numpy(),
                                       "y": sub[head].astype(str).to_numpy(),
                                       "d": sub[DEVICE_COL].to_numpy()})
                         .groupby("k").agg(n_label=("y", "nunique"), n=("y", "count"),
                                           n_dev=("d", "nunique"), label=("y", "first")))
                enough = stats.n >= min_support
                pure = stats[(stats.n_label == 1) & enough & (stats.n_dev >= min_devices)]
                for key, label in pure.label.items():
                    got = index[head].get(str(label))
                    if got is not None:
                        book[str(key)] = int(got)
                for key in stats.index[(stats.n_label > 1) & enough]:
                    book.setdefault(str(key), HIT_AMBIGUOUS)
            tables.setdefault(mode, {})[head] = book
    return tables

## Hierarchy mask và bảng ngưỡng

**Mask nhân vào, không chuẩn hoá lại.** Khi head `model` mâu thuẫn với `make`, phần xác
suất bị cắt phải *biến mất* chứ không được chia lại cho các lớp còn sống. Chuẩn hoá lại
sẽ đẩy confidence lên đúng lúc bằng chứng đang mâu thuẫn, và ngưỡng abstain — thứ duy
nhất chặn ở đây — sẽ không bao giờ kích hoạt.

**Ngưỡng vừa là initializer vừa là graph input.** Tên nằm ở cả `graph.initializer` lẫn
`graph.input` thì initializer đóng vai giá trị mặc định và caller được phép ghi đè lúc
`session.run()`. Nhờ vậy deploy vẫn đúng một file, mà tune ngưỡng ngoài hiện trường chỉ
cần một JSON 2 KB, không rebuild model.

⚠️ Ngưỡng mặc định dưới đây là **giá trị tạm**, chưa calibrate. Số thật phải lấy từ OOF
của `04_evaluate.ipynb`; đo trên dữ liệu in-sample sẽ ra ngưỡng quá lỏng.

In [6]:
def build_hierarchy(frame):
    unique = frame[[MODEL_HEAD, "make", "type"]].astype(str).drop_duplicates()
    return {str(name): {"make": str(g["make"].iloc[0]),
                        "types": sorted(g["type"].unique())}
            for name, g in unique.groupby(MODEL_HEAD, sort=True)}


def hierarchy_masks(hierarchy, labels):
    """[C_model, C_head] nhị phân cho `make` và `type`. Dòng __unknown__ toàn 1."""
    model_classes = [str(c) for c in labels[MODEL_HEAD]]
    masks = {}
    for head in ("make", "type"):
        classes = [str(c) for c in labels[head]]
        index = {c: i for i, c in enumerate(classes)}
        mask = np.ones((len(model_classes), len(classes)), dtype=np.float32)
        for row, name in enumerate(model_classes):
            rule = hierarchy.get(name)
            if not rule or name == UNKNOWN_LABEL:
                continue
            allowed = [rule["make"]] if head == "make" else rule.get("types", [])
            columns = [index[a] for a in allowed if a in index]
            if not columns:
                continue
            mask[row] = 0.0
            mask[row, columns] = 1.0
            if UNKNOWN_LABEL in index:
                mask[row, index[UNKNOWN_LABEL]] = 1.0
        masks[head] = mask
    return masks


def threshold_matrix(thresholds=None):
    source = thresholds or DEFAULT_THRESHOLDS
    rows = []
    for head in HEADS:
        values = [float(v) for v in source[head]]
        assert len(values) == N_SOURCE_BUCKETS, f"{head}: cần {N_SOURCE_BUCKETS} ngưỡng"
        rows.append(values)
    return np.asarray(rows, dtype=np.float32)

## Khối dựng graph

`GB` là bản mở rộng của `OnnxGraphBuilder` ở `07_export_onnx.ipynb`, thêm `add_multi`
cho các op nhiều output (`TopK`).

`str_present` trả về "giá trị này là chuỗi thật" — không phải `''`, `<missing>`, `'nan'`,
`'None'`. Nó thay cho `notna()` của pandas: graph chỉ nhìn thấy ma trận chuỗi, không có
khái niệm NaN.

In [7]:
class GB:
    """Gom node + initializer cho một graph, tự sinh tên không đụng nhau."""

    def __init__(self, prefix="sdc"):
        self.prefix, self.nodes, self.inits, self._n = prefix, [], [], 0

    def name(self, stem):
        self._n += 1
        return f"{self.prefix}/{stem}_{self._n}"

    def add(self, op_type, inputs, out_stem, **attrs):
        out = self.name(out_stem)
        domain = attrs.pop("domain", "")
        self.nodes.append(helper.make_node(op_type, list(inputs), [out],
                                           name=self.name(f"n_{op_type}"),
                                           domain=domain, **attrs))
        return out

    def add_multi(self, op_type, inputs, out_stems, **attrs):
        outs = [self.name(s) for s in out_stems]
        domain = attrs.pop("domain", "")
        self.nodes.append(helper.make_node(op_type, list(inputs), outs,
                                           name=self.name(f"n_{op_type}"),
                                           domain=domain, **attrs))
        return outs

    def const(self, array, stem, dtype=None):
        out = self.name(stem)
        if dtype == TP.STRING:
            values = np.asarray(array, dtype=object)
            self.inits.append(helper.make_tensor(out, TP.STRING, list(values.shape),
                                                 [s.encode("utf-8") for s in values.ravel()]))
        else:
            self.inits.append(onnx.numpy_helper.from_array(np.asarray(array), out))
        return out


def relink(graph, rename):
    nodes = []
    for node in graph.node:
        copy = onnx.NodeProto()
        copy.CopyFrom(node)
        for i, name in enumerate(copy.input):
            if name in rename:
                copy.input[i] = rename[name]
        nodes.append(copy)
    return nodes, list(graph.initializer)


def build_front(gb, cols, num_cols):
    """`input` string [N,len(cols)] -> {tên cột: tensor}. Cột numeric Cast sang float32."""
    outs = [gb.name(f"col/{c}") for c in cols]
    gb.nodes.append(helper.make_node("Split", [IN_FEATURES], outs,
                                     name=gb.name("n_Split"), axis=1, num_outputs=len(cols)))
    num = set(num_cols)
    return {c: (gb.add("Cast", [t], f"num/{c}", to=TP.FLOAT) if c in num else t)
            for c, t in zip(cols, outs)}


def str_present(gb, tensor, stem):
    """bool [N,1]: giá trị là chuỗi thật, không phải '', '<missing>', 'nan', 'None'."""
    bad = None
    for token in NOT_PRESENT:
        eq = gb.add("Equal", [tensor, gb.const(np.array([[token]]), f"tok/{stem}", TP.STRING)],
                    f"eq/{stem}")
        bad = eq if bad is None else gb.add("Or", [bad, eq], f"or/{stem}")
    return gb.add("Not", [bad], f"present/{stem}")


def float_eq(gb, tensor, value, stem):
    return gb.add("Equal", [tensor, gb.const(np.float32([[value]]), f"c/{stem}")], f"eq/{stem}")


def label_encoder(gb, tensor, book, stem, default=-1):
    """ai.onnx.ml LabelEncoder string -> int64. Đây là 'bảng tra' của L0 và L1."""
    keys = list(book.keys())
    values = [int(v) for v in book.values()]
    out = gb.name(f"le/{stem}")
    attrs = {"keys_strings": keys or ["\x00__empty__"],
             "values_int64s": values or [default],
             "default_int64": int(default)}
    gb.nodes.append(helper.make_node("LabelEncoder", [tensor], [out],
                                     name=gb.name("n_LabelEncoder"),
                                     domain="ai.onnx.ml", **attrs))
    return out


def concat_key(gb, tensors, stem):
    """Nối chuỗi bằng StringConcat (opset 20), chèn ' || ' giữa các phần."""
    if len(tensors) == 1:
        return tensors[0]
    sep = gb.const(np.array([[KEY_SEP]]), f"sep/{stem}", TP.STRING)
    out = tensors[0]
    for i, tensor in enumerate(tensors[1:]):
        out = gb.add("StringConcat", [out, sep], f"key/{stem}_{i}s")
        out = gb.add("StringConcat", [out, tensor], f"key/{stem}_{i}v")
    return out

## L0, L1 và khối gộp của một head

Khối gộp là chỗ mọi thứ gặp nhau. Ba điểm đáng chú ý:

**Ứng viên L0 bị gác riêng từng cái.** Gộp trước rồi mới gác thì một nguồn ưu tiên cao
bị hierarchy loại sẽ kéo theo cả câu trả lời hợp lệ của nguồn dưới nó — MAC thật tra ra
`Generic Laptop` trong khi mDNS đã chốt `iPhone`: OUI bị loại (đúng), nhưng hostname
`Johns-iPhone` vốn trả `Apple` cũng biến mất theo (sai).

**Hierarchy đứng trên cả bằng chứng tất định.** Một lớp bị mask cấm không được thắng kể
cả khi L0/L1 chỉ đúng vào nó. Không còn ứng viên nào thì head rơi về xác suất rừng đã
mask và ngưỡng abstain làm việc của nó — mâu thuẫn thành "không biết", không thành hai
câu conf 1.0 chọi nhau.

**Dưới ngưỡng thì nhãn thành `__unknown__`.** Đó là chỗ duy nhất bảng ngưỡng còn tác
dụng khi output chỉ có nhãn. L0/L1 thắng thì confidence là 1.0 nên không bao giờ chạm
ngưỡng — bằng chứng tất định không phải qua cửa này.

In [8]:
def build_l1(gb, columns, tables, head, minus_one):
    """Chuỗi tra vân tay fp_full -> fp_dhcp -> fp_tls. Trả int64 [N,1], -1 = không trả lời.

    -2 (nhập nhằng) dừng chuỗi nhưng không trả lời: mode cụ thể hơn đã không tách được
    nhãn thì mode chung hơn không thể khá hơn.
    """
    answer, decided = minus_one, None
    for mode, cols, flags in FP_MODES:
        book = tables.get(mode, {}).get(head, {})
        key = concat_key(gb, [columns[c] for c in cols], f"{mode}/{head}")
        raw = label_encoder(gb, key, book, f"{mode}/{head}")
        usable = None
        for flag in flags:
            eq = float_eq(gb, columns[flag], 1.0, f"{mode}/{head}/{flag}")
            usable = eq if usable is None else gb.add("And", [usable, eq], f"and/{mode}/{head}")
        for col in cols:
            present = str_present(gb, columns[col], f"{mode}/{head}/{col}")
            usable = gb.add("And", [usable, present], f"and/{mode}/{head}/{col}")
        gated = gb.add("Where", [usable, raw, minus_one], f"l1/{mode}/{head}")
        hit = gb.add("GreaterOrEqual", [gated, gb.const(np.int64([[0]]), f"z/{mode}/{head}")],
                     f"hit/{mode}/{head}")
        stop = gb.add("Less", [gated, minus_one], f"amb/{mode}/{head}")   # -2 < -1
        stop = gb.add("Or", [hit, stop], f"stop/{mode}/{head}")
        if decided is None:
            answer = gb.add("Where", [hit, gated, minus_one], f"ans/{mode}/{head}")
            decided = stop
        else:
            fresh = gb.add("Where", [hit, gated, minus_one], f"ans/{mode}/{head}")
            answer = gb.add("Where", [decided, answer, fresh], f"chain/{mode}/{head}")
            decided = gb.add("Or", [decided, stop], f"dec/{mode}/{head}")
    return answer


def build_l0(gb, columns, resolved, head, minus_one):
    """Các ứng viên L0 theo thứ tự ưu tiên: mDNS model= > OUI > regex hostname.

    Trả một DANH SÁCH tensor int64 [N,1] chứ không gộp sẵn, vì hierarchy phải gác từng
    nguồn một. Gộp trước rồi mới gác thì một nguồn ưu tiên cao bị hierarchy loại sẽ kéo
    theo cả câu trả lời hợp lệ của nguồn dưới nó: MAC thật tra ra `Generic Laptop` trong
    khi mDNS đã chốt `iPhone` -> OUI bị loại (đúng), nhưng hostname `Johns-iPhone` vốn
    trả `Apple` (hợp lệ) cũng biến mất theo, và head `make` rơi xuống rừng một cách vô lý.
    """
    candidates = []

    if head == MODEL_HEAD and resolved["mdns_model"]:
        candidates.append(label_encoder(gb, columns["mdns_model"], resolved["mdns_model"], "mdns"))

    if head == "make" and resolved["oui"]:
        raw = label_encoder(gb, columns["mac_oui"], resolved["oui"], "oui")
        real = float_eq(gb, columns["mac_is_random"], 0.0, "oui/real")
        candidates.append(gb.add("Where", [real, raw, minus_one], "oui/gated"))

    # Luật hostname đứng trước thắng -> dựng ngược để luật đầu nằm ngoài cùng.
    host = None
    for pattern, rule_head, class_idx in reversed(resolved["hostname"]):
        if rule_head != head:
            continue
        match = gb.add("RegexFullMatch", [columns["dhcp_hostname"]], f"rx/{head}", pattern=pattern)
        host = gb.add("Where", [match, gb.const(np.int64([[class_idx]]), f"hv/{head}"),
                                host if host is not None else minus_one], f"host/{head}")
    if host is not None:
        candidates.append(host)

    return candidates


def build_head(gb, proba, class_labels, l0_candidates, l1_idx, thr, minus_one, head,
               allow_row=None):
    """Gộp L0/L1/L2 -> MỘT nhãn `string [N,1]`. Trả dict tensor (`label` là output thật).

    `allow_row` là dòng mask hierarchy [N, C] của head này. Khi có, TỪNG ứng viên bị gác
    riêng: một lớp bị hierarchy cấm không được thắng kể cả khi L0/L1 chỉ đúng vào nó.
    Hierarchy là ràng buộc cứng, đứng trên cả bằng chứng tất định — thiếu gác này thì
    `family=iPhone` (mDNS `model=`) đi cùng `make=Generic Laptop` (OUI) là hai câu trả
    lời conf 1.0 mâu thuẫn nhau mà không tầng nào phía sau nói được là chúng mâu thuẫn.
    Bị cấm thì ứng viên đó rụng, ứng viên ưu tiên thấp hơn được xét tiếp, và nếu không
    còn ai thì head rơi về xác suất rừng đã mask để ngưỡng abstain làm việc của nó.
    """
    n_class = len(class_labels)
    zero_i = gb.const(np.int64([[0]]), f"zero/{head}")

    def gate(tensor, stem):
        if allow_row is None:
            return tensor
        safe = gb.add("Max", [tensor, zero_i], f"safe/{stem}")
        allowed = gb.add("GatherElements", [allow_row, safe], f"allow/{stem}", axis=1)
        allowed = gb.add("Greater", [allowed, gb.const(np.float32([[0.0]]), f"az/{stem}")],
                         f"ok/{stem}")
        return gb.add("Where", [allowed, tensor, minus_one], f"gated/{stem}")

    gated = [gate(c, f"{head}/l0_{i}") for i, c in enumerate(l0_candidates)]
    gated_l1 = gate(l1_idx, f"{head}/l1")

    l0_hit = None
    for i, tensor in enumerate(gated):
        hit = gb.add("GreaterOrEqual", [tensor, zero_i], f"l0hit/{head}_{i}")
        l0_hit = hit if l0_hit is None else gb.add("Or", [l0_hit, hit], f"l0any/{head}_{i}")
    if l0_hit is None:
        l0_hit = gb.add("Less", [zero_i, zero_i], f"l0none/{head}")      # hằng False
    l1_hit = gb.add("GreaterOrEqual", [gated_l1, zero_i], f"l1hit/{head}")

    # Ưu tiên: ứng viên đầu danh sách thắng, L1 đứng sau tất cả L0.
    override = gated_l1
    for i, tensor in reversed(list(enumerate(gated))):
        hit = gb.add("GreaterOrEqual", [tensor, zero_i], f"pick/{head}_{i}")
        override = gb.add("Where", [hit, tensor, override], f"fuse/{head}_{i}")
    has_override = gb.add("GreaterOrEqual", [override, zero_i], f"ovrhit/{head}")
    l1_hit = gb.add("And", [l1_hit, gb.add("Not", [l0_hit], f"nol0/{head}")], f"l1won/{head}")
    layer = gb.add("Where", [l0_hit, gb.const(np.int64([[LAYER_L0]]), f"L0/{head}"),
                             gb.add("Where", [l1_hit, gb.const(np.int64([[LAYER_L1]]), f"L1/{head}"),
                                              gb.const(np.int64([[LAYER_L2]]), f"L2/{head}")],
                                    f"l12/{head}")], f"layer/{head}")

    # Chỉ số thắng của tầng rừng. Khoản trừ TIEBREAK chỉ dùng để xếp hạng nên không ảnh
    # hưởng nhãn, nhưng nó chốt luật phá hoà là của mình chứ không của onnxruntime.
    tie = gb.const((np.arange(n_class, dtype=np.float32) * TIEBREAK).reshape(1, -1), f"tie/{head}")
    ranked = gb.add("Sub", [proba, tie], f"rank/{head}")
    forest_idx = gb.add("ArgMax", [ranked], f"am/{head}", axis=1, keepdims=1)
    forest_conf = gb.add("ReduceMax", [proba, gb.const(np.int64([1]), f"rax/{head}")],
                         f"pmax/{head}", keepdims=1)

    # L0/L1 thắng thì confidence là 1.0 — bằng chứng tất định, không qua ngưỡng.
    one = gb.const(np.float32([[1.0]]), f"one/{head}")
    conf = gb.add("Where", [has_override, one, forest_conf], f"conf/{head}")
    index = gb.add("Where", [has_override, override, forest_idx], f"idx/{head}")

    # Dưới ngưỡng -> trả `__unknown__` thay vì nhãn đoán được.
    abstain = gb.add("Less", [conf, thr], f"abst/{head}")
    unknown = gb.const(np.int64([[class_labels.index(ABSTAIN_LABEL)]]), f"unk/{head}")
    index = gb.add("Where", [abstain, unknown, index], f"final/{head}")

    table = gb.const(np.asarray(class_labels, dtype=object), f"lab/{head}", TP.STRING)
    label = gb.add("Gather", [table, index], f"lab1/{head}", axis=0)
    label = gb.add("Reshape", [label, gb.const(np.int64([-1, 1]), f"rs/{head}")], f"lab2/{head}")

    return {"label": label, "_top1_idx": index, "_layer": layer, "_conf": conf,
            "_abstain": abstain}

## Lắp ráp

Thứ tự quan trọng: head `model` chạy **trước** và lái hierarchy, dùng kết quả **đã gộp
L0/L1** làm driver chứ không dùng `argmax` của rừng. Khi mDNS `model=` nói thẳng đời máy
thì đó mới là câu trả lời của head này, và `make`/`type` phải bị ràng buộc theo nó.

`prune()`, `set_locale()` giữ nguyên từ `07_export_onnx.ipynb` — thiếu `locale="C"` thì
onnxruntime dựng `en_US.UTF-8` lúc nạp model và chết trên musl (router OpenWrt).

In [9]:
def sub_models(bundle):
    enc = bundle["encoder"]
    initial = ([(c, FloatTensorType([None, 1])) for c in enc["num_cols"]]
               + [(c, StringTensorType([None, 1])) for c in enc["cat_cols"] + enc["text_cols"]])
    ct = convert_sklearn(enc["ct"], "sdc_encoder", initial_types=initial, target_opset=ONNX_OPSET)
    ct = compose.add_prefix(ct, "enc/", rename_inputs=False)
    n_feat = len(bundle["feature_names"])
    forests = {}
    for head in HEADS:
        clf = bundle["models"][head]
        m = convert_sklearn(clf, f"sdc_{head}",
                            initial_types=[("X", FloatTensorType([None, n_feat]))],
                            target_opset=ONNX_OPSET, options={id(clf): {"zipmap": False}})
        forests[head] = compose.add_prefix(m, f"rf_{head}/")
    return ct, forests


def prune(model):
    stats = {"weights_kept": 0, "weights_dropped": 0, "attrs_dropped": []}
    for node in model.graph.node:
        if node.op_type not in ("TreeEnsembleClassifier", "TreeEnsembleRegressor"):
            continue
        att = {a.name: a for a in node.attribute}
        weights = np.asarray(att["class_weights"].floats, dtype=np.float32)
        keep = np.flatnonzero(weights != 0)
        stats["weights_kept"] += int(keep.size)
        stats["weights_dropped"] += int(weights.size - keep.size)
        for field in ("class_ids", "class_nodeids", "class_treeids"):
            values = [int(v) for v in np.asarray(att[field].ints)[keep]]
            del att[field].ints[:]
            att[field].ints.extend(values)
        values = [float(v) for v in weights[keep]]
        del att["class_weights"].floats[:]
        att["class_weights"].floats.extend(values)
        hit = att.get("nodes_hitrates")
        miss = att.get("nodes_missing_value_tracks_true")
        drop = ([("nodes_hitrates", hit)] if hit is not None and all(v == 1.0 for v in hit.floats) else [])
        drop += ([("nodes_missing_value_tracks_true", miss)] if miss is not None and all(v == 0 for v in miss.ints) else [])
        for name, attr in drop:
            node.attribute.remove(attr)
            stats["attrs_dropped"].append(name)
    model.graph.ClearField("doc_string")
    for node in model.graph.node:
        node.ClearField("doc_string")
    stats["attrs_dropped"] = sorted(set(stats["attrs_dropped"]))
    return stats


def set_locale(model, locale=ONNX_LOCALE):
    patched = []
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        for existing in [a for a in node.attribute if a.name == "locale"]:
            node.attribute.remove(existing)
        node.attribute.append(helper.make_attribute("locale", locale))
        patched.append(node.name)
    return patched


def assert_locale(model, locale=ONNX_LOCALE):
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        got = {a.name: a for a in node.attribute}.get("locale")
        assert got is not None and got.s.decode() == locale, f"{node.name}: locale sai"


def build_model(bundle, tables, resolved, masks, thresholds, do_prune=True):
    enc = bundle["encoder"]
    base_cols = enc["num_cols"] + enc["cat_cols"] + enc["text_cols"]
    cols = base_cols + list(L0_COLUMNS)
    num_cols = set(enc["num_cols"]) | {"mac_is_random"}
    labels = {h: [str(c) for c in bundle["models"][h].classes_] for h in HEADS}

    ct, forests = sub_models(bundle)
    gb = GB()
    columns = build_front(gb, cols, num_cols)

    # Encoder nối theo VỊ TRÍ (cùng thứ tự với initial_types) — compose.add_prefix vẫn
    # gắn tiền tố vào tên input dù rename_inputs=False.
    nodes, inits = relink(ct.graph, {i.name: columns[c] for i, c in zip(ct.graph.input, base_cols)})
    gb.nodes += nodes
    gb.inits += inits
    features = ct.graph.output[0].name

    proba = {}
    for head in HEADS:
        forest = forests[head]
        nodes, inits = relink(forest.graph, {forest.graph.input[0].name: features})
        gb.nodes += nodes
        gb.inits += inits
        proba[head] = forest.graph.output[1].name        # zipmap=False -> (label, probabilities)

    # --- n_sources -> ngưỡng ---
    total = None
    for flag in SOURCE_FLAGS:
        total = columns[flag] if total is None else gb.add("Add", [total, columns[flag]], "nsrc")
    n_sources = gb.add("Cast", [total], "nsrc_i", to=TP.INT64)
    n_sources = gb.add("Reshape", [n_sources, gb.const(np.int64([-1]), "nsrc_shape")], "nsrc_flat")

    minus_one = gb.const(np.int64([[-1]]), "minus_one")

    def threshold_for(head):
        position = HEADS.index(head)
        row = gb.add("Gather", [IN_THRESHOLDS, gb.const(np.int64(position), f"hi/{head}")],
                     f"thrrow/{head}", axis=0)
        thr = gb.add("Gather", [row, n_sources], f"thr/{head}", axis=0)
        return gb.add("Reshape", [thr, gb.const(np.int64([-1, 1]), f"thrs/{head}")], f"thr2/{head}")

    def fuse(head, proba_tensor, allow_row=None):
        l1 = build_l1(gb, columns, tables, head, minus_one)
        l0 = build_l0(gb, columns, resolved, head, minus_one)
        return build_head(gb, proba_tensor, labels[head], l0, l1, threshold_for(head),
                          minus_one, head, allow_row=allow_row)

    assert masks["make"].shape[0] == len(labels[MODEL_HEAD]), "mask hierarchy lệch số lớp"

    # Head `model` chạy TRƯỚC và lái hierarchy. Dùng kết quả đã gộp L0/L1 làm driver chứ
    # không dùng argmax của rừng: khi mDNS `model=` nói thẳng đời máy, đó mới là câu trả
    # lời của head này, và make/type phải bị ràng buộc theo nó.
    parts = {MODEL_HEAD: fuse(MODEL_HEAD, proba[MODEL_HEAD])}
    driver = gb.add("Reshape", [parts[MODEL_HEAD]["_top1_idx"],
                                gb.const(np.int64([-1]), "drvshape")], "drv")
    for head in ("make", "type"):
        table = gb.const(masks[head], f"mask/{head}")
        row = gb.add("Gather", [table, driver], f"maskrow/{head}", axis=0)
        # Nhân mask, KHÔNG chuẩn hoá lại: phần xác suất bị cắt phải biến mất chứ không
        # được chia lại cho các lớp còn sống, nếu không confidence sẽ tăng lên đúng lúc
        # bằng chứng đang mâu thuẫn và ngưỡng abstain không bao giờ kích hoạt.
        masked = gb.add("Mul", [proba[head], row], f"pm/{head}")
        parts[head] = fuse(head, masked, allow_row=row)

    # Một tensor `string [N,3]`, cột theo đúng thứ tự HEADS.
    gb.nodes.append(helper.make_node("Concat", [parts[h]["label"] for h in HEADS],
                                     [OUT_LABELS], name=gb.name("n_out"), axis=1))

    thr_init = onnx.numpy_helper.from_array(thresholds, IN_THRESHOLDS)
    gb.inits.append(thr_init)

    inputs = [helper.make_tensor_value_info(IN_FEATURES, TP.STRING, [None, len(cols)]),
              # Vừa là input vừa có initializer: initializer là GIÁ TRỊ MẶC ĐỊNH, caller
              # truyền `thresholds` thì ghi đè. Đây là chỗ duy nhất tune được không rebuild.
              helper.make_tensor_value_info(IN_THRESHOLDS, TP.FLOAT,
                                            list(thresholds.shape))]
    outputs = [helper.make_tensor_value_info(OUT_LABELS, TP.STRING, [None, len(HEADS)])]
    graph = helper.make_graph(gb.nodes, "sdc_iden", inputs, outputs, gb.inits)

    opsets = {"": ONNX_OPSET, "ai.onnx.ml": ONNX_ML_OPSET}
    for sub in (ct, *forests.values()):
        for imp in sub.opset_import:
            opsets[imp.domain] = max(opsets.get(imp.domain, 0), imp.version)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid(d, v)
                                                    for d, v in opsets.items()])
    model.ir_version = IR_VERSION
    model.doc_string = ""
    stats = prune(model) if do_prune else {}
    stats["locale_nodes"] = set_locale(model)
    assert_locale(model)
    onnx.checker.check_model(model)
    return model, cols, labels, stats

## Reference bằng numpy/sklearn

Bake 8 tầng vào graph mà không có lưới đối chiếu thì lúc lệch kết quả sẽ không biết tầng
nào gây ra. Hàm dưới đây chạy **đúng cùng một logic** bằng numpy/sklearn, trên **đúng ma
trận chuỗi** mà graph nhận — không phải trên dataframe gốc, vì nếu hai bên nhìn dữ liệu
khác nhau thì phép đối chiếu vô nghĩa.

In [10]:
def prepare_frame(frame, encoder):
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    out = frame[num + cat + text].copy()
    for c in num:
        out[c] = out[c].astype(np.float32)
    for c in cat + text:
        out[c] = out[c].astype(str)
    return out


def encode(frame, encoder):
    return encoder["ct"].transform(prepare_frame(frame, encoder)).astype(np.float32)


def ref_l1(matrix, col_index, tables, head):
    """Bản numpy của chuỗi tra L1 — chạy trên ĐÚNG ma trận chuỗi mà graph nhận."""
    n = len(matrix)
    answer = np.full(n, -1, dtype=np.int64)
    decided = np.zeros(n, dtype=bool)
    for mode, cols, flags in FP_MODES:
        book = tables.get(mode, {}).get(head, {})
        usable = np.ones(n, dtype=bool)
        for flag in flags:
            usable &= matrix[:, col_index[flag]].astype(np.float32) == 1.0
        for col in cols:
            values = matrix[:, col_index[col]].astype(str)
            usable &= ~np.isin(values, NOT_PRESENT)
        keys = np.array([KEY_SEP.join(str(r[col_index[c]]) for c in cols) for r in matrix])
        raw = np.array([book.get(k, -1) for k in keys], dtype=np.int64)
        raw = np.where(usable, raw, -1)
        fresh = np.where(raw >= 0, raw, -1)
        answer = np.where(decided, answer, fresh)
        decided = decided | (raw != -1)
    return answer


def ref_l0(matrix, col_index, resolved, head):
    """Danh sách ứng viên L0 theo thứ tự ưu tiên — khớp `build_l0`, chưa gộp."""
    n = len(matrix)
    candidates = []
    if head == MODEL_HEAD and resolved["mdns_model"]:
        value = matrix[:, col_index["mdns_model"]].astype(str)
        candidates.append(np.array([resolved["mdns_model"].get(v, -1) for v in value],
                                   dtype=np.int64))
    if head == "make" and resolved["oui"]:
        oui = matrix[:, col_index["mac_oui"]].astype(str)
        real = matrix[:, col_index["mac_is_random"]].astype(np.float32) == 0.0
        raw = np.array([resolved["oui"].get(o, -1) for o in oui], dtype=np.int64)
        candidates.append(np.where(real, raw, -1))
    host, hostnames = None, matrix[:, col_index["dhcp_hostname"]].astype(str)
    for pattern, rule_head, class_idx in reversed(resolved["hostname"]):
        if rule_head != head:
            continue
        match = np.array([re.fullmatch(pattern, h) is not None for h in hostnames])
        host = np.where(match, class_idx, host if host is not None else np.full(n, -1, np.int64))
    if host is not None:
        candidates.append(host)
    return candidates


def reference(frame, matrix, cols, bundle, tables, resolved, masks, thresholds):
    """Toàn bộ pipeline bằng numpy/sklearn — lưới an toàn để đối chiếu với graph.

    Trả `output` [N,3] (đúng bằng output của graph) CỘNG các trường chẩn đoán tiền tố `_`:
    confidence, tầng nào trả lời, có abstain không. Graph không lộ mấy trường đó ra
    ngoài, nhưng bài kiểm L0 cần chúng để nói được "đường này có chạy không" — thiếu thì
    chỉ kiểm được nhãn cuối, và một đường L0 chết vẫn có thể cho ra nhãn đúng nhờ rừng.
    """
    col_index = {c: i for i, c in enumerate(cols)}
    labels = {h: np.array([str(c) for c in bundle["models"][h].classes_]) for h in HEADS}
    X = encode(frame, bundle["encoder"])
    proba = {h: bundle["models"][h].predict_proba(X).astype(np.float32) for h in HEADS}

    n_sources = sum(matrix[:, col_index[f]].astype(np.float32) for f in SOURCE_FLAGS).astype(int)

    def fuse(head, p, allow_row=None):
        def gate(values):
            if allow_row is None:
                return values
            allowed = allow_row[np.arange(len(values)), np.maximum(values, 0)] > 0
            return np.where(allowed, values, -1)

        gated = [gate(c) for c in ref_l0(matrix, col_index, resolved, head)]
        gated_l1 = gate(ref_l1(matrix, col_index, tables, head))

        l0_hit = np.zeros(len(p), dtype=bool)
        for candidate in gated:
            l0_hit |= candidate >= 0
        override = gated_l1
        for candidate in reversed(gated):
            override = np.where(candidate >= 0, candidate, override)
        has_override = override >= 0
        l1_hit = (gated_l1 >= 0) & ~l0_hit
        layer = np.where(l0_hit, LAYER_L0, np.where(l1_hit, LAYER_L1, LAYER_L2))

        ranked = p - np.arange(p.shape[1], dtype=np.float32) * TIEBREAK
        forest_idx = ranked.argmax(axis=1)
        conf = np.where(has_override, 1.0, p.max(axis=1)).astype(np.float32)
        index = np.where(has_override, override, forest_idx)

        thr = thresholds[HEADS.index(head)][n_sources]
        abstain = conf < thr
        index = np.where(abstain, list(labels[head]).index(ABSTAIN_LABEL), index)
        return {"label": labels[head][index], "_top1_idx": index, "_layer": layer,
                "_conf": conf, "_abstain": abstain}

    parts = {MODEL_HEAD: fuse(MODEL_HEAD, proba[MODEL_HEAD])}
    driver = parts[MODEL_HEAD]["_top1_idx"]
    for head in ("make", "type"):
        row = masks[head][driver]
        parts[head] = fuse(head, proba[head] * row, allow_row=row)

    result = {OUT_LABELS: np.stack([parts[h]["label"] for h in HEADS], axis=1)}
    for key in ("_layer", "_conf", "_abstain"):
        result[key] = np.stack([parts[h][key] for h in HEADS], axis=1)
    return result

## Build

Mọi bảng tra được đào ở đây rồi nung thành initializer. `metadata_props` mang theo
contract đầy đủ cộng hai hash: thứ tự cột và thứ tự nhãn.

Hash cột là thứ chặn lỗi nguy hiểm nhất của bản deploy này: **đủ 44 cột nhưng sai thứ
tự**. Shape mismatch thì onnxruntime chửi ngay — đó là trường hợp may mắn. Sai thứ tự thì
không exception nào bắn ra, chỉ là accuracy tụt và không ai biết tại sao.

In [11]:
def latest_run(pattern="*_field_closedset"):
    runs = sorted(MODELS.glob(pattern))
    assert runs, f"Không thấy run nào khớp {pattern} trong {MODELS}"
    return runs[-1]


def build_all(run_dir=None, out_dir=None, thresholds=None):
    run_dir = Path(run_dir or latest_run())
    bundle = joblib.load(run_dir / "model.joblib")
    assert bundle["format"] == "sdc-closedset-v1", f"format lạ: {bundle['format']}"

    frame = add_l0_columns(pd.read_parquet(SESSIONS_PATH))
    labels = {h: [str(c) for c in bundle["models"][h].classes_] for h in HEADS}

    rules = build_rules(frame)
    resolved = resolve_rules(rules, labels)
    tables = mine_fingerprints(frame, labels)
    hierarchy = build_hierarchy(frame)
    masks = hierarchy_masks(hierarchy, labels)
    thr = threshold_matrix(thresholds)

    model, cols, labels, stats = build_model(bundle, tables, resolved, masks, thr)
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_iden")
    meta = {
        "format": FORMAT, "contract_version": CONTRACT_VERSION, "run_id": run_id,
        "source_run": run_dir.name,
        "columns": cols, "columns_sha256": sha256_list(cols),
        "labels": labels,
        "labels_sha256": sha256_list([f"{h}:{c}" for h in HEADS for c in labels[h]]),
        "heads": list(HEADS), "output": OUT_LABELS, "abstain_label": ABSTAIN_LABEL,
        "hierarchy": hierarchy,
        "thresholds": {h: thr[i].tolist() for i, h in enumerate(HEADS)},
        "rules_dropped": resolved["dropped"],
        "sizes": {"l1": {f"{m}/{h}": len(b) for m, hs in tables.items() for h, b in hs.items()},
                  "oui": len(resolved["oui"]), "mdns_model": len(resolved["mdns_model"]),
                  "hostname": len(resolved["hostname"])},
        "size_optimization": stats,
    }
    for key, value in (("sdc.contract", json.dumps(meta, ensure_ascii=False)),
                       ("sdc.columns_sha256", meta["columns_sha256"]),
                       ("sdc.labels_sha256", meta["labels_sha256"]),
                       ("sdc.run_id", run_id), ("sdc.format", FORMAT)):
        model.metadata_props.append(onnx.StringStringEntryProto(key=key, value=value))

    out_dir = Path(out_dir or (MODELS / run_id))
    out_dir.mkdir(parents=True, exist_ok=True)
    payload = model.SerializeToString()
    (out_dir / ONNX_FILE).write_bytes(payload)
    meta["bytes"] = len(payload)
    return {"model": model, "path": out_dir / ONNX_FILE, "meta": meta, "frame": frame,
            "bundle": bundle, "tables": tables, "resolved": resolved, "masks": masks,
            "thresholds": thr, "cols": cols}


built = build_all()
print(f"{built['path']}  {built['meta']['bytes']:,} bytes")
print(json.dumps(built["meta"]["sizes"], indent=2, ensure_ascii=False))
print("rules bị bỏ:", built["meta"]["rules_dropped"])

D:\01.AI_Security\02.SDC\Models\20260917_154635_iden\sdc_iden.onnx  1,618,076 bytes
{
  "l1": {
    "fp_full/make": 2,
    "fp_full/type": 2,
    "fp_full/model": 2,
    "fp_dhcp/make": 5,
    "fp_dhcp/type": 5,
    "fp_dhcp/model": 5,
    "fp_tls/make": 3,
    "fp_tls/type": 3,
    "fp_tls/model": 3
  },
  "oui": 6,
  "mdns_model": 9,
  "hostname": 14
}
rules bị bỏ: ["make='Amazon'", "make='Amcrest'", "make='Arlo'", "make='Borun'", "make='D-Link'", "make='Eufy'", "make='Google'", "make='HeimVision'", "make='Home Eye'", "make='LG'", "make='Luohe'", "make='Netatmo'", "make='Philips'", "make='Ring'", "make='SimCam'", "make='Sonos'", "make='Tuya ODM'", "make='iRobot'"]


## Đối chiếu graph ↔ reference

Output chỉ còn nhãn nên phép so là so chuỗi, đúng/sai rạch ròi. Bảng tách theo từng cột
của `output` để biết head nào lệch, chứ tensor thì chỉ có một. `TIEBREAK` vẫn cần: xác
suất của rừng có rất nhiều giá trị bằng nhau, và `ArgMax` của onnxruntime không bảo đảm
phá hoà giống `argmax` của numpy. Trừ một lượng tăng dần theo chỉ số lớp trước khi xếp
hạng biến luật phá hoà thành của mình (chỉ số nhỏ thắng) thay vì của bản onnxruntime
đang chạy trên router.

`reference()` trả thêm mấy trường chẩn đoán có tiền tố `_` (confidence, tầng nào trả
lời). Graph không lộ chúng ra, nhưng bài kiểm L0 ở cell sau cần — chỉ so nhãn cuối thì
một đường L0 chết vẫn có thể cho nhãn đúng nhờ rừng, và bài kiểm sẽ báo PASS nhầm.

In [12]:
def verify(built, limit=None, seed=0):
    """Đối chiếu `output` [N,3] của graph với reference numpy/sklearn.

    Output chỉ còn nhãn nên phép so là so chuỗi, đúng/sai rạch ròi — không còn cột 'hoà'
    như khi còn topk/retrieval: hoà xếp hạng chỉ đổi thứ tự ứng viên phía sau, mà những
    ứng viên đó không còn được trả ra nữa.
    """
    frame, cols = built["frame"], built["cols"]
    if limit and limit < len(frame):
        frame = frame.sample(limit, random_state=seed)
    matrix = to_input_matrix(frame, cols)
    want = reference(frame, matrix, cols, built["bundle"], built["tables"],
                     built["resolved"], built["masks"], built["thresholds"])
    session = ort.InferenceSession(str(built["path"]), providers=["CPUExecutionProvider"])
    got = dict(zip([o.name for o in session.get_outputs()],
                   session.run(None, {IN_FEATURES: matrix})))
    want_a = np.asarray(want[OUT_LABELS]).astype(str)
    got_a = np.asarray(got[OUT_LABELS]).astype(str)
    report = [{"cột": f"output[:, {i}]  {head}",
               "lệch": int((want_a[:, i] != got_a[:, i]).sum()), "trên": len(want_a)}
              for i, head in enumerate(HEADS)]
    return pd.DataFrame(report).set_index("cột"), want, got


parity, want, got = verify(built)
print(f"Đối chiếu {len(built['frame']):,} dòng — graph ONNX so với reference numpy/sklearn")
print(f"output: string {np.asarray(got[OUT_LABELS]).shape}")
print(parity.to_string())
assert parity["lệch"].sum() == 0, "graph lệch reference"
print()
print("PARITY OK")

Đối chiếu 8,189 dòng — graph ONNX so với reference numpy/sklearn
output: string (8189, 3)
                     lệch  trên
cột                            
output[:, 0]  make      0  8189
output[:, 1]  type      0  8189
output[:, 2]  model     0  8189

PARITY OK


## Ép ba đường L0 chạy

Dataset hiện tại chưa có `dhcp_hostname` và `mdns_model`, nên cả ba đường L0 **không
được chạm** ở bài đối chiếu bên trên — nó vẫn báo PASS trong khi L0 là code chết. Cell
này dựng dòng tổng hợp để ép chúng chạy, kèm một ca xung đột cố ý.

In [13]:
# Dataset hiện tại chưa có `dhcp_hostname` và `mdns_model` (extractor chưa trích), nên cả
# ba đường L0 đều không được chạm ở bài đối chiếu trên. Dựng dòng tổng hợp để ép chúng
# chạy — không có cell này thì L0 là code chết mà vẫn báo PASS.
def l0_probe(built):
    """Sáu dòng tổng hợp, mỗi dòng ép đúng một đường L0 chạy.

    Hai dòng gốc khác nhau, cố ý: `plain` là một dòng FIELD bị xoá OUI (giả lập MAC ngẫu
    nhiên hoá) để thử hostname/mDNS một mình; `owned` là một dòng có OUI THẬT nằm trong
    bảng, để thử đường OUI và để dựng ca xung đột trên chính nó.
    """
    frame = built["frame"]
    table = built["resolved"]["oui"]
    field = frame[frame["scenario"].eq("FIELD")]
    plain = (field if len(field) else frame).iloc[[0]].copy()
    owned = frame[frame["mac_oui"].isin(table)]
    assert len(owned), "không có dòng nào mang OUI nằm trong bảng — không thử được L0/OUI"
    owned = owned.iloc[[0]].copy()
    rows = []

    def case(name, base, **over):
        row = base.copy()
        for key, value in over.items():
            row[key] = value
        row.index = [name]
        rows.append(row)

    case("hostname=Johns-iPhone", plain, dhcp_hostname="Johns-iPhone",
         mac_oui=MISSING, mac_is_random=1)
    case("hostname=DESKTOP-AB12CD3", plain, dhcp_hostname="DESKTOP-AB12CD3",
         mac_oui=MISSING, mac_is_random=1)
    case("mdns model=iPhone17,1", plain, mdns_model="iPhone17,1",
         mac_oui=MISSING, mac_is_random=1)
    case("OUI thật", owned)
    case("MAC ngẫu nhiên", owned, mac_oui=MISSING, mac_is_random=1)
    case("XUNG ĐỘT mdns vs OUI", owned, mdns_model="iPhone17,1")
    return pd.concat(rows)


probe = l0_probe(built)
probe_matrix = to_input_matrix(probe, built["cols"])
session = ort.InferenceSession(str(built["path"]), providers=["CPUExecutionProvider"])
out = dict(zip([o.name for o in session.get_outputs()],
               session.run(None, {IN_FEATURES: probe_matrix})))
ref = reference(probe, probe_matrix, built["cols"], built["bundle"], built["tables"],
                built["resolved"], built["masks"], built["thresholds"])

# Graph chỉ trả nhãn; tầng nào trả lời thì đọc từ reference (đã đối chiếu khớp ở trên).
labels = out[OUT_LABELS]
print(pd.DataFrame({
    "make": labels[:, 0],
    "type": labels[:, 1],
    "model": labels[:, 2],
    "tầng": [" / ".join(LAYER_NAMES[int(v)].split("_")[0] for v in r) for r in ref["_layer"]],
    "conf": [" / ".join(f"{v:.3f}" for v in r) for r in ref["_conf"]],
}, index=probe.index).to_string())

assert (np.asarray(ref[OUT_LABELS]).astype(str) == labels.astype(str)).all()
layer = ref["_layer"]
assert (layer[0] == LAYER_L0).all(), "regex hostname không kích hoạt cả 3 head"
assert layer[1][1] == LAYER_L0, "DESKTOP- không kích hoạt head type"
assert layer[2][2] == LAYER_L0, "mDNS model= không kích hoạt head model"
assert layer[3][0] == LAYER_L0, "OUI không kích hoạt head make"
assert (layer[4] != LAYER_L0).all(), "MAC ngẫu nhiên vẫn tra được OUI"

# Ca xung đột: mDNS chốt `model=iPhone17,1`, còn OUI của MAC thật trỏ sang hãng khác.
# Hierarchy phải loại override của OUI — head make không được trả nhãn OUI ở conf 1.0.
oui_label = str(labels[3, 0])              # nhãn mà đường OUI trả ở dòng 'OUI thật'
assert layer[5][0] != LAYER_L0, "xung đột: override OUI vẫn lọt qua hierarchy"
assert str(labels[5, 0]) != oui_label, f"xung đột: make vẫn trả {oui_label}"
assert str(labels[5, 2]) == "iPhone", "xung đột: mDNS phải thắng ở head model"
print()
print("Sáu đường L0 đều đúng như thiết kế")

                                    make         type               model          tầng                   conf
hostname=Johns-iPhone              Apple   Smartphone              iPhone  L0 / L0 / L0  1.000 / 1.000 / 1.000
hostname=DESKTOP-AB12CD3  Generic Laptop       Laptop  Windows Desktop HP  L2 / L0 / L2  1.000 / 1.000 / 1.000
mdns model=iPhone17,1        __unknown__  __unknown__              iPhone  L2 / L2 / L0  0.000 / 0.000 / 1.000
OUI thật                  Generic Laptop       Laptop  Windows Desktop HP  L0 / L2 / L2  1.000 / 1.000 / 1.000
MAC ngẫu nhiên            Generic Laptop       Laptop  Windows Desktop HP  L2 / L2 / L2  1.000 / 1.000 / 1.000
XUNG ĐỘT mdns vs OUI         __unknown__  __unknown__              iPhone  L2 / L2 / L0  0.000 / 0.000 / 1.000

Sáu đường L0 đều đúng như thiết kế


## Ngưỡng ghi đè lúc chạy

In [14]:
# Ngưỡng vừa là initializer vừa là graph input. Không truyền -> dùng mặc định nung trong
# file; truyền -> ghi đè. Đây là thứ duy nhất tune được ngoài hiện trường mà không rebuild.
# Bỏ output `abstain` nên hiệu lực của ngưỡng đọc qua số nhãn `__unknown__`.
sample = to_input_matrix(built["frame"].sample(200, random_state=1), built["cols"])


def n_unknown(thresholds=None):
    feeds = {IN_FEATURES: sample}
    if thresholds is not None:
        feeds[IN_THRESHOLDS] = thresholds
    out = ort.InferenceSession(str(built["path"]), providers=["CPUExecutionProvider"]).run(
        [OUT_LABELS], feeds)[0]
    return int((np.asarray(out).astype(str) == UNKNOWN_LABEL).sum()), out.size


base, total = n_unknown()
always, _ = n_unknown(np.full((3, N_SOURCE_BUCKETS), 1.01, np.float32))
never, _ = n_unknown(np.zeros((3, N_SOURCE_BUCKETS), np.float32))
print(f"ngưỡng mặc định : __unknown__ {base:4d}/{total}")
print(f"ngưỡng 1.01     : __unknown__ {always:4d}/{total}   (1.01 > 1.0 nên kể cả conf 1.0 "
      "cũng bị chặn — công tắc 'luôn từ chối')")
print(f"ngưỡng 0.00     : __unknown__ {never:4d}/{total}   (chỉ còn những ca rừng TỰ nói "
      "không biết)")
assert always == total and never <= base
print()
print("Override ngưỡng lúc chạy OK — không cần build lại model")

ngưỡng mặc định : __unknown__  594/600
ngưỡng 1.01     : __unknown__  600/600   (1.01 > 1.0 nên kể cả conf 1.0 cũng bị chặn — công tắc 'luôn từ chối')
ngưỡng 0.00     : __unknown__  594/600   (chỉ còn những ca rừng TỰ nói không biết)

Override ngưỡng lúc chạy OK — không cần build lại model


## Runtime — phần chạy trên router

`IdenSession` + `device_log` là toàn bộ code phía thiết bị ngoài extractor. `assert_extractor`
so hash cột trước khi chạy và **từ chối chạy** nếu lệch, thay vì chạy sai trong im lặng.

Log dùng tên `family` cho head thứ ba; bundle gọi nó là `model` từ contract 2.2 — ánh xạ
nằm ở `LOG_HEAD_NAMES`.

In [15]:
# Log dùng tên `family` cho head thứ ba; bundle gọi nó là `model` từ contract 2.2.
LOG_HEAD_NAMES = {"make": "make", "type": "type", "model": "family"}


class IdenSession:
    """Bọc onnxruntime -> JSON log. Toàn bộ phần chạy trên router, ngoài extractor.

    Graph chỉ trả ba nhãn, nên log cũng chỉ có ba trường nhận dạng. Muốn biết vì sao ra
    nhãn đó (tầng nào trả lời, confidence bao nhiêu) thì phải build lại model với output
    đầy đủ — thông tin ấy không tồn tại ngoài graph nữa.
    """

    def __init__(self, path, thresholds=None):
        self.path = Path(path)
        self.session = ort.InferenceSession(str(self.path), providers=["CPUExecutionProvider"])
        self.meta = json.loads({p.key: p.value for p in
                                onnx.load(str(self.path)).metadata_props}["sdc.contract"])
        self.columns = self.meta["columns"]
        self.thresholds = None if thresholds is None else np.asarray(thresholds, np.float32)

    def assert_extractor(self, columns):
        """Chốt thứ tự cột trước khi chạy. Đủ 44 cột nhưng SAI THỨ TỰ là lỗi im lặng:
        không exception nào bắn ra, chỉ là accuracy tụt và không ai biết tại sao."""
        got = sha256_list(columns)
        if got != self.meta["columns_sha256"]:
            raise RuntimeError(f"extractor lệch contract: {got[:16]} != "
                               f"{self.meta['columns_sha256'][:16]} — từ chối chạy")

    def run(self, matrix):
        feeds = {IN_FEATURES: matrix}
        if self.thresholds is not None:
            feeds[IN_THRESHOLDS] = self.thresholds
        return self.session.run([self.meta["output"]], feeds)[0]

    def window_logs(self, frame):
        self.assert_extractor(self.columns)
        matrix = to_input_matrix(frame, self.columns)
        out = self.run(matrix)
        version = {"model_version": self.meta["run_id"],
                   "contract_version": f"{self.meta['format']}/{self.meta['contract_version']}"}
        logs = []
        for i in range(len(matrix)):
            log = {"mac": str(frame["mac"].iloc[i]) if "mac" in frame else MISSING,
                   "dhcp_hostname": str(matrix[i][self.columns.index("dhcp_hostname")])}
            for position, head in enumerate(HEADS):
                log[LOG_HEAD_NAMES[head]] = str(out[i][position])
            log.update(version)
            logs.append(log)
        return logs


def device_log(window_logs):
    """Gộp nhiều cửa sổ của cùng một MAC — bỏ phiếu đa số, bỏ qua `__unknown__`.

    Đây là phần state duy nhất KHÔNG nhét được vào graph: graph ONNX là stateless, gộp
    theo MAC là state xuyên lời gọi. Nó ở lại dưới dạng code, mãi mãi.

    `__unknown__` bị loại khỏi phiếu vì nó vừa mang nghĩa "rừng nói không biết" vừa mang
    nghĩa "dưới ngưỡng" — gộp nó vào phiếu thì một thiết bị lạ và một thiết bị quen thu
    được cửa sổ xấu trông giống hệt nhau. Không cửa sổ nào trả lời được thì mới kết luận
    `__unknown__`.
    """
    from collections import Counter
    if not window_logs:
        return {}
    out = {"mac": window_logs[0]["mac"], "dhcp_hostname": window_logs[0]["dhcp_hostname"],
           "n_windows": len(window_logs)}
    for head in HEADS:
        name = LOG_HEAD_NAMES[head]
        votes = Counter(w[name] for w in window_logs if w[name] != UNKNOWN_LABEL)
        if not votes:
            out[name] = UNKNOWN_LABEL
            out[f"{name}_vote"] = f"0/{len(window_logs)}"
            continue
        label, n = votes.most_common(1)[0]
        out[name] = label
        out[f"{name}_vote"] = f"{n}/{len(window_logs)}"
    for key in ("model_version", "contract_version"):
        out[key] = window_logs[0][key]
    return out


runtime = IdenSession(built["path"])
print("output của graph:", [(o.name, o.type, o.shape) for o in runtime.session.get_outputs()])
print("thứ tự cột:", runtime.meta["heads"])
print()

_demo = built["frame"][built["frame"]["scenario"].eq("FIELD")].head(3).copy()
_demo["dhcp_hostname"] = "Johns-iPhone"
_demo["mdns_model"] = ["iPhone17,1", MISSING, MISSING]
_logs = runtime.window_logs(_demo)
print(json.dumps(_logs[0], indent=2, ensure_ascii=False))
print()
print("--- gộp theo MAC ---")
print(json.dumps(device_log(_logs), indent=2, ensure_ascii=False))

output của graph: [('output', 'tensor(string)', [None, 3])]
thứ tự cột: ['make', 'type', 'model']

{
  "mac": "50:bb:b5:fc:5b:c2",
  "dhcp_hostname": "Johns-iPhone",
  "make": "Apple",
  "type": "Smartphone",
  "family": "iPhone",
  "model_version": "20260917_154635_iden",
  "contract_version": "sdc-iden-onnx-v1/1.0.0"
}

--- gộp theo MAC ---
{
  "mac": "50:bb:b5:fc:5b:c2",
  "dhcp_hostname": "Johns-iPhone",
  "n_windows": 3,
  "make": "Apple",
  "make_vote": "3/3",
  "type": "Smartphone",
  "type_vote": "3/3",
  "family": "iPhone",
  "family_vote": "3/3",
  "model_version": "20260917_154635_iden",
  "contract_version": "sdc-iden-onnx-v1/1.0.0"
}


## Test trên bộ pcap tự thu

Ba nguồn trong `Data/` ghép lại thành khung 44 cột:

| File | Cho gì |
|---|---|
| `features/session_features_capture.csv` | 40 cột feature, 62 cửa sổ / 15 MAC / 6 file pcap |
| `features/dhcp_features_capture.csv` | cột `hostname` — **DHCP option 12 thật** |
| `data_pcap/device_labels_capture.csv` | nhãn make/type/model theo hostname |

Đây là lần đầu `dhcp_hostname` có giá trị thật thay vì `<missing>`, nên cũng là lần đầu
đường L0/hostname chạy trên dữ liệu không do mình bịa. `mdns_model` vẫn `<missing>`: bộ
capture này không có bản ghi TXT `_device-info._tcp` nào (0/976 dòng mDNS) — nó có
`_companion-link`, `_rdlink`, `_sleep-proxy` là dấu hiệu Apple, nhưng đó là hướng khác.

In [16]:
FEAT_DIR = DATA / "features"
PCAP_DIR = DATA / "data_pcap"


def load_capture():
    """Dựng khung 44 cột từ bộ pcap tự thu, kèm nhãn ground-truth.

    Ba nguồn ghép lại:
      - `session_features_capture.csv` : 40 cột feature, một dòng mỗi (MAC, file pcap)
      - `dhcp_features_capture.csv`    : cột `hostname` = DHCP option 12 **thật**
      - `device_labels_capture.csv`    : nhãn make/type/model theo hostname

    Đây là lần đầu cột `dhcp_hostname` có giá trị thật thay vì `<missing>`, nên cũng là
    lần đầu đường L0/hostname được chạy trên dữ liệu không phải do mình bịa ra.
    `mdns_model` vẫn `<missing>`: bộ capture này không có bản ghi TXT `_device-info._tcp`
    nào (đã kiểm: 0/976 dòng mDNS).
    """
    sess = pd.read_csv(FEAT_DIR / "session_features_capture.csv")
    dhcp = pd.read_csv(FEAT_DIR / "dhcp_features_capture.csv")
    hostnames = (dhcp.dropna(subset=["hostname"]).drop_duplicates("mac")
                 .set_index("mac")["hostname"])
    sess["dhcp_hostname"] = sess["mac"].map(hostnames).fillna(MISSING)
    sess["mdns_model"] = MISSING
    sess["canonical_device"] = sess["device"]

    truth = pd.read_csv(PCAP_DIR / "device_labels_capture.csv").set_index("hostname")
    frame = add_l0_columns(sess)
    for head in HEADS:
        frame[f"truth_{head}"] = frame["device"].map(truth[head])
    missing = [c for c in built["cols"] if c not in frame.columns]
    assert not missing, f"capture thiếu cột: {missing}"
    return frame


capture = load_capture()
cap_matrix = to_input_matrix(capture, built["cols"])
cap_out = ort.InferenceSession(str(built["path"]), providers=["CPUExecutionProvider"]).run(
    [OUT_LABELS], {IN_FEATURES: cap_matrix})[0]
cap_ref = reference(capture, cap_matrix, built["cols"], built["bundle"], built["tables"],
                    built["resolved"], built["masks"], built["thresholds"])
assert (np.asarray(cap_ref[OUT_LABELS]).astype(str) == cap_out.astype(str)).all()

for position, head in enumerate(HEADS):
    capture[f"pred_{head}"] = cap_out[:, position]
capture["layer"] = [" / ".join(LAYER_NAMES[int(v)].split("_")[0] for v in r)
                    for r in cap_ref["_layer"]]

print(f"{len(capture)} cửa sổ / {capture['mac'].nunique()} MAC / "
      f"{capture['capture_file'].nunique()} file pcap")
print(f"hostname DHCP đọc được: {(capture['dhcp_hostname'] != MISSING).sum()}/{len(capture)} dòng")
print(f"OUI tra được          : {(capture['mac_oui'] != MISSING).sum()}/{len(capture)} dòng "
      f"({(capture['mac_is_random'] == 1).sum()} dòng MAC ngẫu nhiên hoá)")

62 cửa sổ / 15 MAC / 6 file pcap
hostname DHCP đọc được: 47/62 dòng
OUI tra được          : 37/62 dòng (25 dòng MAC ngẫu nhiên hoá)


In [17]:
# ⚠️ 12 MAC có nhãn CHÍNH LÀ tập FIELD đã train (cùng 6 file pcap). Phần này đo trí nhớ,
# không đo khả năng khái quát — sai ở đây là lỗi thật, còn đúng ở đây không chứng minh
# được gì. Phần thật sự held-out là 3 MAC `Unknown-*` ở cell sau: chúng bị loại khỏi
# `sessions_verified.parquet` nên model chưa từng thấy.
labeled = capture[capture["truth_make"].notna()]
rows = []
for device, group in labeled.groupby("device", sort=True):
    row = {"thiết bị": device, "n": len(group),
           "hostname": group["dhcp_hostname"].iloc[0],
           "tầng": group["layer"].iloc[0]}
    for head in HEADS:
        want = group[f"truth_{head}"].iloc[0]
        got = group[f"pred_{head}"]
        top = got.mode().iloc[0]
        hit = int((got == want).sum())
        mark = "" if top == want else f" ✗ (đúng: {want})"
        row[head] = f"{top} [{hit}/{len(group)}]{mark}"
    rows.append(row)
print(pd.DataFrame(rows).set_index("thiết bị").to_string())

print()
for head in HEADS:
    ok = (labeled[f"pred_{head}"] == labeled[f"truth_{head}"]).sum()
    print(f"  {head:6s} đúng {ok:3d}/{len(labeled)} cửa sổ")

                         n             hostname          tầng                  make                         type                       model
thiết bị                                                                                                                                    
DESKTOP-DNGJHRT          5      desktop-dngjhrt  L0 / L0 / L2  Generic Laptop [5/5]                 Laptop [5/5]    Windows Desktop HP [5/5]
DESKTOP-DucAnh           6       desktop-ducanh  L0 / L0 / L2  Generic Laptop [6/6]                 Laptop [6/6]  Windows Desktop DELL [6/6]
DESKTOP-cua-Lee-KingDom  5          lee_kingdom  L0 / L2 / L2  Generic Laptop [5/5]                 Laptop [5/5]  Windows Desktop DELL [5/5]
IP-Camera                6            <missing>  L0 / L2 / L2          Camera [6/6]              IP Camera [6/6]     Generic IP Camera [6/6]
LAPTOP-VAF70SQ6          6            <missing>  L0 / L2 / L2  Generic Laptop [6/6]                 Laptop [6/6]     Windows Laptop HP [6/6]
OPPO-A92     

### Phần held-out thật

Ba MAC `Unknown-*` không có trong bảng nhãn nên bị loại khỏi `sessions_verified.parquet`
— model chưa từng thấy chúng. 12 cửa sổ này mới là bài đo thật, và nó đo đúng cái mà
kiến trúc L0 sinh ra để làm.

In [18]:
# 3 MAC không có trong bảng nhãn -> bị loại khỏi tập train. Đây mới là bài đo thật.
unseen = capture[capture["truth_make"].isna()]
print(f"{len(unseen)} cửa sổ của {unseen['mac'].nunique()} MAC chưa từng đưa vào lúc train")
print()
for mac, group in unseen.groupby("mac", sort=True):
    logs = [{"mac": mac, "dhcp_hostname": group["dhcp_hostname"].iloc[0],
             **{LOG_HEAD_NAMES[h]: v for h, v in zip(HEADS, row)},
             "model_version": "", "contract_version": ""}
            for row in zip(*[group[f"pred_{h}"] for h in HEADS])]
    voted = device_log(logs)
    print(f"{mac}  hostname={group['dhcp_hostname'].iloc[0]!r}  ({len(group)} cửa sổ, "
          f"MAC {'ngẫu nhiên' if group['mac_is_random'].iloc[0] else 'thật'})")
    print(f"   -> make={voted['make']}  type={voted['type']}  family={voted['family']}"
          f"   [{group['layer'].iloc[0]}]")
print()
print("Hai MAC mang hostname 'iphone' là phép thử đáng giá nhất ở đây: model chưa từng")
print("thấy chúng, và nếu L0 nhận ra được thì đó là công của luật hostname chứ không")
print("phải của rừng.")

12 cửa sổ của 3 MAC chưa từng đưa vào lúc train

06:27:04:ee:2a:40  hostname='iphone'  (4 cửa sổ, MAC ngẫu nhiên)
   -> make=Apple  type=Smartphone  family=iPhone   [L0 / L0 / L0]
16:76:ae:9b:f1:a0  hostname='iphone'  (6 cửa sổ, MAC ngẫu nhiên)
   -> make=Apple  type=Smartphone  family=iPhone   [L0 / L0 / L0]
6e:4c:77:fc:06:d3  hostname='<missing>'  (2 cửa sổ, MAC ngẫu nhiên)
   -> make=__unknown__  type=Smartphone  family=__unknown__   [L2 / L2 / L2]

Hai MAC mang hostname 'iphone' là phép thử đáng giá nhất ở đây: model chưa từng
thấy chúng, và nếu L0 nhận ra được thì đó là công của luật hostname chứ không
phải của rừng.


### Luật hostname nào thực sự nổ

Luật không bao giờ khớp là luật chết — nó chỉ tồn tại trong đầu người viết, và chỉ dữ
liệu thu thật mới chỉ ra được.

Bộ capture này đã bắt được hai lỗi trong bảng luật mặc định, cả hai đều đã sửa ở cell
`rules` phía trên:

1. `DESKTOP-[A-Z0-9]{7}` **không khớp gì cả**. DHCP option 12 về tới nơi đã bị hạ chữ
   thường (`desktop-ducanh`, `desktop-dngjhrt`), và phần đuôi không phải lúc nào cũng
   đúng 7 ký tự. Đổi thành `(?i)desktop-.+`.
2. Thiếu hẳn luật cho `samsung-*`, dù đã có `oppo-*` và `redmi-*` cùng dạng. Gộp vào
   luật `galaxy`.

⚠️ Hai sửa đổi này được lái bởi chính bộ capture đang dùng để chấm điểm, nên bảng kết
quả phía trên **không** còn là bằng chứng độc lập cho chúng.

In [19]:
# Luật hostname nào thực sự khớp trên hostname THẬT. Luật không bao giờ nổ là luật chết —
# nó chỉ tồn tại trong đầu người viết, và chỉ dữ liệu thu thật mới chỉ ra được.
seen = sorted(set(capture.loc[capture["dhcp_hostname"] != MISSING, "dhcp_hostname"]))
rows = []
for pattern, head, class_idx in built["resolved"]["hostname"]:
    matched = [h for h in seen if re.fullmatch(pattern, h)]
    rows.append({"pattern": pattern, "head": head,
                 "nhãn": built["meta"]["labels"][head][class_idx],
                 "khớp": len(matched), "ví dụ": ", ".join(matched[:3]) or "—"})
print(pd.DataFrame(rows).to_string(index=False))
print()
unmatched = [h for h in seen
             if not any(re.fullmatch(p, h) for p, _, _ in built["resolved"]["hostname"])]
print(f"hostname không luật nào bắt được ({len(unmatched)}/{len(seen)}):", unmatched)

                       pattern  head                  nhãn  khớp                           ví dụ
                (?i)desktop-.+  type                Laptop     2 desktop-dngjhrt, desktop-ducanh
                 (?i)laptop-.+  type                Laptop     0                               —
                  (?i).*iphone  make                 Apple     1                          iphone
                  (?i).*iphone  type            Smartphone     1                          iphone
                  (?i).*iphone model                iPhone     1                          iphone
                    (?i).*ipad  make                 Apple     0                               —
               (?i).*macbook.*  make                 Apple     0                               —
        (?i)raspberrypi[0-9-]*  make          Raspberry Pi     1                     raspberrypi
        (?i)raspberrypi[0-9-]*  type Single-board Computer     1                     raspberrypi
        (?i)raspberrypi[0-9-]*

## Feature importance của 44 cột

Rừng **không** học trên 44 cột mà trên 1088 feature: TF-IDF nở `dns_tokens` thành 575
cột, `tls_sni_tokens` 327, `mdns_tokens` 118, `dhcp_vci` 32. `feature_names` đặt tên
chúng là `<cột nguồn>::<token>` nên gộp về cột nguồn chỉ là cộng theo tiền tố.

Cộng thẳng như vậy có một thiên lệch phải đọc kèm: cột nở ra 575 feature gần như chắc
chắn gom nhiều importance hơn cột chỉ có 1, kể cả khi mỗi token riêng lẻ yếu hơn. Cột
`trên_1_feature` là để bù cho chuyện đó — và nó đảo ngược thứ hạng: `tls_ciphers` mạnh
gấp ~39 lần `dns_tokens` nếu tính trên mỗi feature.

⚠️ Điều đáng lo nhất nằm ở hai dòng đầu bảng. `dns_tokens` + `tls_sni_tokens` chiếm
**~77%** importance, mà đó đúng là hai cột `03_train_model.ipynb` tự đánh dấu
`ENCRYPTED_RISK`. DoH/DoT làm biến mất cột thứ nhất, ECH làm biến mất cột thứ hai — và
khi đó model mất 3/4 thứ nó đang dựa vào. Nhóm DHCP (26 cờ option) cộng lại chỉ 2,48%, và ba cột text chiếm 85,2%.

In [20]:
def forest_importance(built):
    """Gộp importance của 1088 feature đã encode về 40 cột nguồn.

    Rừng không học trên 44 cột mà trên 1088 feature: TF-IDF nở `dns_tokens` thành 575
    cột, `tls_sni_tokens` thành 327, `mdns_tokens` 118, `dhcp_vci` 32. `feature_names`
    đặt tên chúng là `<cột nguồn>::<token>`, nên gộp lại chỉ là cộng theo tiền tố.

    Cộng thẳng như vậy có một thiên lệch phải biết: một cột nở ra 575 feature gần như
    chắc chắn gom được nhiều importance hơn một cột chỉ có 1, kể cả khi mỗi token riêng
    lẻ yếu hơn. Cột `trên_1_feature` ở bảng dưới là để đọc kèm — nó là importance trung
    bình của mỗi feature con.
    """
    bundle = built["bundle"]
    source = pd.Series([f.split("::")[0] for f in bundle["feature_names"]])
    table = pd.DataFrame({
        head: pd.Series(bundle["models"][head].feature_importances_).groupby(source).sum()
        for head in HEADS
    })
    table["trung_bình"] = table.mean(axis=1)
    table["n_feature"] = source.value_counts()
    table["trên_1_feature"] = table["trung_bình"] / table["n_feature"]
    return table.sort_values("trung_bình", ascending=False)


def l0_coverage(frame, built):
    """Tỉ lệ dòng mà mỗi nguồn L0 đưa ra được câu trả lời.

    Bốn cột 41-44 KHÔNG có trong rừng nên importance của chúng bằng 0 theo định nghĩa —
    con số đó vô nghĩa, không phải là "không quan trọng". Thứ đo được cho chúng là độ
    phủ: bao nhiêu phần trăm dòng thì bảng tra / luật regex của chúng nổ.
    """
    matrix = to_input_matrix(frame, built["cols"])
    index = {c: i for i, c in enumerate(built["cols"])}
    resolved = built["resolved"]
    n = len(matrix)

    real_mac = matrix[:, index["mac_is_random"]].astype(np.float32) == 0.0
    oui = matrix[:, index["mac_oui"]].astype(str)
    oui_hit = np.array([resolved["oui"].get(v, -1) >= 0 for v in oui]) & real_mac

    mdns = matrix[:, index["mdns_model"]].astype(str)
    mdns_hit = np.array([resolved["mdns_model"].get(v, -1) >= 0 for v in mdns])

    host = matrix[:, index["dhcp_hostname"]].astype(str)
    host_hit = np.zeros(n, dtype=bool)
    for pattern, _, _ in resolved["hostname"]:
        host_hit |= np.array([re.fullmatch(pattern, v) is not None for v in host])

    return pd.DataFrame([
        {"cột": "mac_oui", "có giá trị": int((oui != MISSING).sum()),
         "trả lời được": int(oui_hit.sum()), "%": 100 * oui_hit.sum() / n,
         "nuôi tầng": "L0 OUI -> head make"},
        {"cột": "mac_is_random", "có giá trị": n,
         "trả lời được": int((~real_mac).sum()), "%": 100 * (~real_mac).sum() / n,
         "nuôi tầng": "cổng chặn mac_oui (số dòng BỊ chặn)"},
        {"cột": "dhcp_hostname", "có giá trị": int((host != MISSING).sum()),
         "trả lời được": int(host_hit.sum()), "%": 100 * host_hit.sum() / n,
         "nuôi tầng": "L0 regex -> cả 3 head"},
        {"cột": "mdns_model", "có giá trị": int((mdns != MISSING).sum()),
         "trả lời được": int(mdns_hit.sum()), "%": 100 * mdns_hit.sum() / n,
         "nuôi tầng": "L0 mDNS -> head model"},
    ]).set_index("cột")


def bar(value, scale, width=28):
    return "#" * int(round(width * value / scale)) if scale else ""


imp = forest_importance(built)
print("=== 40 cột qua encoder — importance của rừng (%), gộp từ 1088 feature ===")
print(f"{'cột':16s} {'make':>7s} {'type':>7s} {'model':>7s} {'TB':>7s}  {'n_feat':>6s}")
top = imp["trung_bình"].max()
for name, row in imp.iterrows():
    print(f"{name:16s} {100*row['make']:6.2f}% {100*row['type']:6.2f}% "
          f"{100*row['model']:6.2f}% {100*row['trung_bình']:6.2f}%  {int(row['n_feature']):6d}  "
          f"{bar(row['trung_bình'], top)}")
print(f"{'TỔNG':16s} {100*imp['make'].sum():6.2f}% {100*imp['type'].sum():6.2f}% "
      f"{100*imp['model'].sum():6.2f}% {100*imp['trung_bình'].sum():6.2f}%  "
      f"{int(imp['n_feature'].sum()):6d}")

=== 40 cột qua encoder — importance của rừng (%), gộp từ 1088 feature ===
cột                 make    type   model      TB  n_feat
dns_tokens        45.16%  37.02%  41.91%  41.37%     575  ############################
tls_sni_tokens    33.64%  33.59%  39.59%  35.61%     327  ########################
mdns_tokens        9.43%   9.32%   7.79%   8.85%     118  ######
tls_ciphers        2.68%   3.44%   2.39%   2.83%       1  ##
tls_fp             2.01%   3.54%   1.56%   2.37%       1  ##
tls_alpn           1.41%   2.26%   1.35%   1.67%       1  #
has_tls            0.79%   3.29%   0.85%   1.64%       1  #
tls_version        0.97%   2.36%   0.75%   1.36%       1  #
dhcp_vci           1.38%   0.37%   1.23%   0.99%      32  #
has_dns            0.61%   0.53%   0.61%   0.58%       1  
dhcp_prl_len       0.14%   0.56%   0.23%   0.31%       1  
dhcp_prl           0.26%   0.47%   0.17%   0.30%       1  
has_mdns           0.17%   0.30%   0.10%   0.19%       1  
dhcp_opt_26        0.09%   0.26%   0

### 4 cột L0

Importance của chúng bằng 0 theo **định nghĩa** — chúng không đi qua encoder nên không
có trong rừng. Con số đó không có nghĩa là chúng vô dụng, chỉ có nghĩa là phải đo bằng
thước khác: bao nhiêu phần trăm dòng thì bảng tra / luật regex của chúng nổ.

Đo trên hai bộ vì chúng khác hẳn nhau: `sessions_verified.parquet` chưa có hostname và
mDNS TXT nên ba đường L0 gần như im lặng; bộ capture tự thu mới cho thấy độ phủ thật.

In [21]:
# 4 cột L0 không nằm trong rừng nên importance của chúng bằng 0 theo ĐỊNH NGHĨA, không
# phải vì chúng vô dụng. Đo bằng độ phủ trên hai bộ dữ liệu khác nhau mới ra được bức
# tranh thật: `sessions_verified` chưa có hostname/mDNS, bộ capture tự thu thì có.
for label, frame in (("sessions_verified.parquet", built["frame"]),
                     ("bộ capture tự thu", capture)):
    print(f"=== {label} ({len(frame)} dòng) ===")
    print(l0_coverage(frame, built).to_string(
        formatters={"%": "{:.1f}%".format}))
    print()

=== sessions_verified.parquet (8189 dòng) ===
               có giá trị  trả lời được    %                            nuôi tầng
cột                                                                              
mac_oui              8176            37 0.5%                  L0 OUI -> head make
mac_is_random        8189            13 0.2%  cổng chặn mac_oui (số dòng BỊ chặn)
dhcp_hostname           0             0 0.0%                L0 regex -> cả 3 head
mdns_model              0             0 0.0%                L0 mDNS -> head model

=== bộ capture tự thu (62 dòng) ===
               có giá trị  trả lời được     %                            nuôi tầng
cột                                                                               
mac_oui                37            37 59.7%                  L0 OUI -> head make
mac_is_random          62            25 40.3%  cổng chặn mac_oui (số dòng BỊ chặn)
dhcp_hostname          47            36 58.1%                L0 regex -> cả 3 head
mdns_model

### Xuất CSV

Ghi cạnh model, encoding `utf-8-sig` để Excel mở không vỡ tiếng Việt. Bốn cột L0 nằm
cuối file với `nguồn = "L0 rule"` và độ phủ ghi kèm, để không bị đọc nhầm là importance
bằng 0.

In [22]:
out_path = built["path"].parent / "feature_importance.csv"
export = imp.copy()
for head in HEADS:
    export[head] = 100 * export[head]
export["trung_bình"] = 100 * export["trung_bình"]
export["trên_1_feature"] = 100 * export["trên_1_feature"]
export.index.name = "cột"
export["nguồn"] = ["L2 rừng"] * len(export)

l0 = l0_coverage(capture, built)
l0_rows = pd.DataFrame({
    "make": np.nan, "type": np.nan, "model": np.nan, "trung_bình": np.nan,
    "n_feature": 0, "trên_1_feature": np.nan,
    "nguồn": "L0 rule — độ phủ " + l0["%"].round(1).astype(str) + "% trên bộ capture",
}, index=l0.index)
l0_rows.index.name = "cột"

pd.concat([export, l0_rows]).round(4).to_csv(out_path, encoding="utf-8-sig")
print("đã ghi", out_path)
print(pd.read_csv(out_path).head(8).to_string(index=False))

đã ghi D:\01.AI_Security\02.SDC\Models\20260917_154635_iden\feature_importance.csv
           cột    make    type   model  trung_bình  n_feature  trên_1_feature   nguồn
    dns_tokens 45.1633 37.0220 41.9132     41.3662        575          0.0719 L2 rừng
tls_sni_tokens 33.6406 33.5887 39.5948     35.6080        327          0.1089 L2 rừng
   mdns_tokens  9.4268  9.3247  7.7855      8.8457        118          0.0750 L2 rừng
   tls_ciphers  2.6788  3.4353  2.3881      2.8340          1          2.8340 L2 rừng
        tls_fp  2.0081  3.5371  1.5648      2.3700          1          2.3700 L2 rừng
      tls_alpn  1.4082  2.2582  1.3451      1.6705          1          1.6705 L2 rừng
       has_tls  0.7890  3.2914  0.8481      1.6429          1          1.6429 L2 rừng
   tls_version  0.9738  2.3592  0.7519      1.3616          1          1.3616 L2 rừng


### Xuất PNG

Hai panel, **không dùng chung trục và không thể dùng chung**: panel trên là importance
(tổng 100% mỗi head), panel dưới là tỉ lệ dòng mà một luật nổ. Vẽ chồng lên một trục sẽ
ngụ ý hai con số so sánh được với nhau, mà chúng thì không.

Bảng màu là 3 slot đầu của palette mặc định (blue / orange / aqua), đã chạy qua
validator ở chế độ light, nền `#fcfcfb`, kiểm **toàn bộ** cặp — PASS. Riêng aqua có WARN
contrast 2.74 < 3.0 nên **bắt buộc** dán nhãn giá trị trực tiếp; đó là lý do con số nằm
ngay cạnh thanh chứ không chỉ có trên trục. Tên series cũng được dán thẳng vào thanh của
hàng đầu, để định danh không phụ thuộc riêng vào màu.

Máy này không có `node` nên validator được port sang Python (giữ nguyên ngưỡng và ma
trận Machado-Oliveira-Fernandes 2009 của bản gốc).

In [23]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Bảng màu đã chạy qua validator (bản Python của scripts/validate_palette.js, vì máy
# này không có node): 3 slot đầu của palette mặc định, chế độ light, nền #fcfcfb, kiểm
# TOÀN BỘ cặp. Kết quả PASS; riêng aqua có WARN contrast 2.74 < 3.0 nên bắt buộc dán
# nhãn giá trị trực tiếp — đó là lý do cột số nằm ngay cạnh thanh chứ không chỉ có trục.
SERIES = {"make": "#2a78d6", "type": "#eb6834", "model": "#1baf7a"}
SURFACE = "#fcfcfb"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
})


def importance_png(built, imp, capture, min_pct=0.0025):
    """Hai panel: importance của rừng, và độ phủ của 4 cột L0.

    Hai panel KHÔNG dùng chung trục và không thể dùng chung: panel trên là importance
    (tổng 100% mỗi head), panel dưới là tỉ lệ dòng mà một luật nổ. Vẽ chồng chúng lên
    một trục sẽ ngụ ý hai con số so sánh được với nhau, mà chúng thì không.
    """
    big = imp[imp["trung_bình"] >= min_pct].copy()
    rest = imp[imp["trung_bình"] < min_pct]
    rows = big.index.tolist()
    values = {h: big[h].tolist() for h in HEADS}
    if len(rest):
        rows.append(f"{len(rest)} cột còn lại (gộp)")
        for head in HEADS:
            values[head].append(rest[head].sum())

    y = np.arange(len(rows))[::-1]          # hạng 1 nằm trên cùng
    height, offset = 0.22, 0.27
    fig, (ax, ax2) = plt.subplots(
        2, 1, figsize=(11.5, 9.0), dpi=150,
        gridspec_kw={"height_ratios": [len(rows), 3.2], "hspace": 0.20})

    # --- panel trên ---------------------------------------------------------------
    for k, head in enumerate(HEADS):
        ax.barh(y + (1 - k) * offset, [100 * v for v in values[head]], height=height,
                color=SERIES[head], linewidth=0, label=head, zorder=3)
        for pos, value in zip(y + (1 - k) * offset, values[head]):
            if 100 * value >= 1.0:          # nhãn chọn lọc, không dán lên mọi thanh
                ax.text(100 * value + 0.45, pos, f"{100*value:.1f}", va="center",
                        ha="left", fontsize=7.5, color=INK2, zorder=4)

    ax.set_yticks(y, rows, fontsize=9, color=INK2)
    ax.set_xlim(0, 50)
    ax.set_xticks(range(0, 50, 10), [f"{v}%" for v in range(0, 50, 10)],
                  fontsize=8.5, color=MUTED)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(BASELINE)
    ax.tick_params(length=0)
    ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=INK2,
              handlelength=1.1, handleheight=0.9)
    # Nhãn series trực tiếp NẰM TRONG thanh của hàng đầu: định danh không phụ thuộc
    # riêng vào màu, mà cũng không tràn ra ngoài khung như khi đặt bên phải.
    for k, head in enumerate(HEADS):
        ax.text(100 * values[head][0] - 1.0, y[0] + (1 - k) * offset, head,
                va="center", ha="right", fontsize=8, color=SURFACE, weight="bold")
    ax.set_title("Importance của rừng — 40 cột đi qua encoder (tổng 100% mỗi head)",
                 loc="left", fontsize=11, color=INK, pad=10)

    # --- panel dưới ---------------------------------------------------------------
    cover = l0_coverage(capture, built).drop(index="mac_is_random")
    order = cover["%"].sort_values(ascending=True)
    y2 = np.arange(len(order))
    ax2.barh(y2, order.values, height=0.32, color=SERIES["make"], linewidth=0, zorder=3)
    ax2.set_ylim(-0.6, len(order) - 0.4)
    for pos, (name, value) in zip(y2, order.items()):
        note = "  (extractor chưa trích)" if value == 0 else ""
        ax2.text(value + 0.8, pos, f"{value:.1f}%{note}", va="center", ha="left",
                 fontsize=8.5, color=INK2, zorder=4)
    ax2.set_yticks(y2, order.index, fontsize=9, color=INK2)
    ax2.set_xlim(0, 100)
    ax2.set_xticks(range(0, 101, 25), [f"{v}%" for v in range(0, 101, 25)],
                   fontsize=8.5, color=MUTED)
    ax2.xaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax2.set_axisbelow(True)
    for side in ("top", "right", "bottom"):
        ax2.spines[side].set_visible(False)
    ax2.spines["left"].set_color(BASELINE)
    ax2.tick_params(length=0)
    ax2.set_title("4 cột L0 — không có trong rừng, đo bằng ĐỘ PHỦ trên bộ capture "
                  f"({len(capture)} cửa sổ)", loc="left", fontsize=11, color=INK, pad=10)

    fig.suptitle("SDC — feature importance, 44 cột", x=0.008, y=0.992, ha="left",
                 fontsize=14, color=INK, weight="bold")
    fig.text(0.008, 0.948,
             "3 cột text chiếm 85,2%; riêng dns_tokens + tls_sni_tokens là 77,2% — đúng "
             "hai cột project tự đánh dấu ENCRYPTED_RISK.",
             ha="left", fontsize=9, color=INK2)
    fig.text(0.008, 0.928,
             "DoH/DoT xoá cột thứ nhất, ECH xoá cột thứ hai. 26 cờ DHCP option cộng lại "
             "chỉ 2,5%.",
             ha="left", fontsize=9, color=INK2)
    fig.text(0.008, 0.012,
             f"run {built['meta']['run_id']}  ·  importance gộp từ 1088 feature đã "
             "encode theo tiền tố `<cột>::<token>`  ·  cột nở nhiều token gom được "
             "nhiều importance hơn, xem `trên_1_feature` trong CSV",
             ha="left", fontsize=7.5, color=MUTED)
    fig.subplots_adjust(left=0.165, right=0.985, top=0.895, bottom=0.055)

    path = built["path"].parent / "feature_importance.png"
    fig.savefig(path)
    plt.close(fig)
    return path


png_path = importance_png(built, imp, capture)
print("đã ghi", png_path, f"({png_path.stat().st_size:,} bytes)")

đã ghi D:\01.AI_Security\02.SDC\Models\20260917_154635_iden\feature_importance.png (152,899 bytes)


## Cái gì THẬT SỰ quyết định đầu ra

Cell importance ở trên chỉ nói được chuyện bên trong rừng. Nó không biết rằng phần lớn
thời gian rừng còn **chưa được hỏi ý kiến** — L0 đã chốt xong từ trước.

Bảng dưới đây quy cả ba tầng về **một thước đo duy nhất**: phần trăm quyết định cuối
cùng do từng nguồn định đoạt. Cộng lại đúng 100% mỗi head. Phần thuộc về rừng được chia
tiếp cho 40 cột theo importance nội bộ của nó.

⚠️ Phép chia đó là **quy kết theo tỉ lệ, không phải nhân quả từng dòng**: importance là
thống kê toàn cục của cây, không phải đóng góp của một cột vào đúng một quyết định cụ
thể. Bốn dòng không phải cột — `dhcp_hostname`/`mac_oui`/`mdns_model` là khoá tra L0,
`vân tay L1` là khoá ghép ba cột, `ngưỡng abstain` là policy — được để nguyên vì chúng
quyết định trực tiếp, không qua rừng.

Chênh lệch giữa hai bộ dữ liệu là toàn bộ câu chuyện: trên `sessions_verified` (chưa có
hostname) bảng này gần như trùng với bảng importance; trên bộ capture thật thì L0 chiếm
hơn nửa.

In [24]:
# MỘT thước đo dùng chung cho cả 44 cột: phần trăm QUYẾT ĐỊNH CUỐI CÙNG do thứ này định
# đoạt. L0, L1 và rừng quy về cùng đơn vị nên cộng lại đúng 100% mỗi head.
#
# Khác hẳn feature importance ở cell trước. Importance trả lời "trong rừng, cột nào tách
# dữ liệu tốt nhất" — nó không biết rằng phần lớn thời gian rừng còn chưa được hỏi ý
# kiến, vì L0 đã chốt xong từ trước. Bảng dưới đây mới là thứ mô tả đầu ra thật.
#
# Một chỗ phải nói rõ: phần của rừng được CHIA cho 40 cột theo importance nội bộ. Đó là
# quy kết theo tỉ lệ, không phải quy kết nhân quả từng dòng — importance là thống kê toàn
# cục của cây, không phải đóng góp của cột đó vào đúng quyết định đó.
L1_ROW = "vân tay L1 (dhcp_prl+dhcp_vci+tls_fp)"
ABSTAIN_ROW = "ngưỡng abstain → __unknown__"


def l0_names(resolved, head):
    """Tên ứng viên L0 theo ĐÚNG thứ tự `build_l0` dựng ra."""
    names = []
    if head == MODEL_HEAD and resolved["mdns_model"]:
        names.append("mdns_model")
    if head == "make" and resolved["oui"]:
        names.append("mac_oui")
    if any(rh == head for _, rh, _ in resolved["hostname"]):
        names.append("dhcp_hostname")
    return names


def decision_source(frame, built):
    """Trả {head: mảng tên nguồn đã quyết định} — phản chiếu `reference()`."""
    cols = built["cols"]
    matrix = to_input_matrix(frame, cols)
    index = {c: i for i, c in enumerate(cols)}
    bundle, resolved = built["bundle"], built["resolved"]
    tables, masks, thr = built["tables"], built["masks"], built["thresholds"]
    X = encode(frame, bundle["encoder"])
    proba = {h: bundle["models"][h].predict_proba(X).astype(np.float32) for h in HEADS}
    n_src = sum(matrix[:, index[f]].astype(np.float32) for f in SOURCE_FLAGS).astype(int)

    out, idx_out = {}, {}

    def fuse(head, p, allow_row=None):
        def gate(v):
            if allow_row is None:
                return v
            ok = allow_row[np.arange(len(v)), np.maximum(v, 0)] > 0
            return np.where(ok, v, -1)

        cands = [gate(c) for c in ref_l0(matrix, index, resolved, head)]
        names = l0_names(resolved, head)
        assert len(cands) == len(names), (len(cands), names)
        l1 = gate(ref_l1(matrix, index, tables, head))

        source = np.full(len(p), "rừng (L2)", dtype=object)
        override = np.where(l1 >= 0, l1, -1)
        source = np.where(l1 >= 0, L1_ROW, source)
        for cand, name in zip(reversed(cands), reversed(names)):
            override = np.where(cand >= 0, cand, override)
            source = np.where(cand >= 0, name, source)

        has = override >= 0
        ranked = p - np.arange(p.shape[1], dtype=np.float32) * TIEBREAK
        conf = np.where(has, 1.0, p.max(axis=1)).astype(np.float32)
        final = np.where(has, override, ranked.argmax(axis=1))
        abstain = conf < thr[HEADS.index(head)][n_src]
        source = np.where(abstain, ABSTAIN_ROW, source)
        return source, np.where(abstain, -1, final)

    src, fin = fuse(MODEL_HEAD, proba[MODEL_HEAD])
    out[MODEL_HEAD], idx_out[MODEL_HEAD] = src, fin
    driver = np.where(fin >= 0, fin, proba[MODEL_HEAD].argmax(axis=1))
    for head in ("make", "type"):
        row = masks[head][driver]
        out[head], _ = fuse(head, proba[head] * row, allow_row=row)
    return out


def attribution(frame, built, imp):
    """% quyết định cuối. Phần của rừng được CHIA theo importance nội bộ của nó."""
    src = decision_source(frame, built)
    table = {}
    for head in HEADS:
        share = pd.Series(src[head]).value_counts(normalize=True)
        forest = share.pop("rừng (L2)") if "rừng (L2)" in share.index else 0.0
        col = imp[head] / imp[head].sum() * forest        # chia phần rừng theo importance
        table[head] = col.add(share, fill_value=0.0)
    return pd.DataFrame(table).fillna(0.0)




for label, frame in (("bộ capture tự thu", capture), ("sessions_verified", built["frame"])):
    att = attribution(frame, built, imp)
    att["trung_bình"] = att.mean(axis=1)
    print(f"=== {label} ({len(frame)} dòng × 3 head) ===")
    print((100 * att.sort_values("trung_bình", ascending=False)).round(2).head(12).to_string())
    print()

=== bộ capture tự thu (62 dòng × 3 head) ===
                                        make   type  model  trung_bình
dhcp_hostname                          35.48  43.55  25.81       34.95
mac_oui                                59.68   0.00   0.00       19.89
dns_tokens                              0.73  17.32  29.74       15.93
tls_sni_tokens                          0.54  15.71  28.10       14.78
mdns_tokens                             0.15   4.36   5.53        3.35
vân tay L1 (dhcp_prl+dhcp_vci+tls_fp)   0.00   8.06   0.00        2.69
ngưỡng abstain → __unknown__            3.23   1.61   3.23        2.69
tls_ciphers                             0.04   1.61   1.69        1.11
tls_fp                                  0.03   1.65   1.11        0.93
has_tls                                 0.01   1.54   0.60        0.72
tls_alpn                                0.02   1.06   0.95        0.68
tls_version                             0.02   1.10   0.53        0.55

=== sessions_verified (8189 dòn

### PNG

Một panel, một trục, một đơn vị — khác với `feature_importance.png` phải tách hai panel
vì importance và độ phủ không so sánh được với nhau.

In [25]:
def attribution_png(built, imp, capture, min_pct=0.0025):
    att = attribution(capture, built, imp)
    att["trung_bình"] = att.mean(axis=1)
    att = att.sort_values("trung_bình", ascending=False)
    big, rest = att[att["trung_bình"] >= min_pct], att[att["trung_bình"] < min_pct]
    rows = big.index.tolist()
    values = {h: big[h].tolist() for h in HEADS}
    if len(rest):
        rows.append(f"{len(rest)} cột còn lại (gộp)")
        for head in HEADS:
            values[head].append(rest[head].sum())

    y = np.arange(len(rows))[::-1]
    fig, ax = plt.subplots(figsize=(11.5, 0.52 * len(rows) + 2.4), dpi=150)
    for k, head in enumerate(HEADS):
        ax.barh(y + (1 - k) * 0.27, [100 * v for v in values[head]], height=0.22,
                color=SERIES[head], linewidth=0, label=head, zorder=3)
        for pos, value in zip(y + (1 - k) * 0.27, values[head]):
            if 100 * value >= 2.0:
                ax.text(100 * value + 0.6, pos, f"{100*value:.1f}", va="center",
                        ha="left", fontsize=7.5, color=INK2, zorder=4)
    for k, head in enumerate(HEADS):
        ax.text(100 * values[head][0] - 1.2, y[0] + (1 - k) * 0.27, head, va="center",
                ha="right", fontsize=8, color=SURFACE, weight="bold")

    ax.set_yticks(y, rows, fontsize=9, color=INK2)
    ax.set_xlim(0, 68)
    ax.set_xticks(range(0, 70, 10), [f"{v}%" for v in range(0, 70, 10)],
                  fontsize=8.5, color=MUTED)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(BASELINE)
    ax.tick_params(length=0)
    ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=INK2,
              handlelength=1.1, handleheight=0.9)

    fig.suptitle("SDC — cái gì thật sự quyết định đầu ra", x=0.008, y=0.985,
                 ha="left", fontsize=14, color=INK, weight="bold")
    fig.text(0.008, 0.945,
             "% quyết định cuối cùng do từng nguồn định đoạt, trên bộ capture tự thu "
             f"({len(capture)} cửa sổ). Cộng lại đúng 100% mỗi head.",
             ha="left", fontsize=9, color=INK2)
    fig.text(0.008, 0.918,
             "Luật L0 chốt ~55% đầu ra. dns_tokens tụt từ 41% (importance) xuống 14,7% "
             "(quyết định thật) — phần lớn thời gian rừng chưa kịp được hỏi.",
             ha="left", fontsize=9, color=INK2)
    fig.text(0.008, 0.018,
             f"run {built['meta']['run_id']}  ·  phần của rừng được chia cho 40 cột theo "
             "importance nội bộ — quy kết theo tỉ lệ, không phải nhân quả từng dòng",
             ha="left", fontsize=7.5, color=MUTED)
    fig.subplots_adjust(left=0.28, right=0.985, top=0.885, bottom=0.10)

    path = built["path"].parent / "decision_attribution.png"
    fig.savefig(path)
    plt.close(fig)
    return path


attrib_png_path = attribution_png(built, imp, capture)
print("đã ghi", attrib_png_path, f"({attrib_png_path.stat().st_size:,} bytes)")

đã ghi D:\01.AI_Security\02.SDC\Models\20260917_154635_iden\decision_attribution.png (114,289 bytes)
